# 003c Exact matched-ratio top-m decoding heatmaps

This notebook makes top-m decoding heatmaps using **only exact matched component ratios**.

Filtering logic:

1. Keep only `K` values that exist in both **within** and **across** for each analysis type.
2. Convert to normalized component ratio:
   - spatial: `K / 700`
   - temporal: `K / 300`
3. Keep only ratios that are exactly shared between spatial and temporal.
4. Plot only those matched ratios. No interpolation is performed.

This means each column corresponds to a true matched-ratio pair, e.g. spatial `K=70` and temporal `K=30` both correspond to ratio `0.10`.

In [ ]:
from pathlib import Path
from fractions import Fraction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# ============================================================
# User settings
# ============================================================

BASE_DIR = Path(".")  # directory containing msaa_condrank_decoding_outputs_* folders

FIG_DIR = Path("figures") / "003c_exact_matched_ratio_topm_heatmaps"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TABLE_DIR = Path("tables") / "003c_exact_matched_ratio_topm_heatmaps"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

CONDITIONS = ["intact", "word", "rest"]
TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 40, 50, 75, 100]

DIMENSIONS = {
    "spatial": 700,
    "temporal": 300,
}

K_VALUES = {
    ("temporal", "within"): [2, 3, 5, 6, 9, 10, 15, 18, 23, 25, 27, 30, 38, 45, 50, 54, 60, 75, 81, 90, 100, 108, 129, 171, 214, 257, 300],
    ("temporal", "across"): [2, 3, 5, 6, 8, 9, 10, 11, 14, 15, 17, 18, 20, 23, 25, 27, 30, 35, 38, 40, 45, 50, 54, 55, 60, 65, 75, 81, 90, 100, 108, 129, 171, 200, 214, 257, 300],
    ("spatial", "across"): [5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50, 52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100, 105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500, 600, 700],
    ("spatial", "within"): [5, 7, 10, 14, 21, 25, 35, 42, 50, 53, 63, 70, 75, 88, 100, 105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500, 600, 700],
}

CMAP = "viridis"

In [ ]:
# ============================================================
# Compute exact matched ratio table
# ============================================================

def common_within_across_k(analysis):
    return sorted(set(K_VALUES[(analysis, "within")]).intersection(K_VALUES[(analysis, "across")]))

spatial_common_k = common_within_across_k("spatial")
temporal_common_k = common_within_across_k("temporal")

spatial_ratio_to_k = {Fraction(k, DIMENSIONS["spatial"]): k for k in spatial_common_k}
temporal_ratio_to_k = {Fraction(k, DIMENSIONS["temporal"]): k for k in temporal_common_k}

matched_ratios = sorted(set(spatial_ratio_to_k).intersection(temporal_ratio_to_k))

matched_ratio_df = pd.DataFrame([
    {
        "ratio_fraction": f"{r.numerator}/{r.denominator}",
        "component_ratio": float(r),
        "spatial_K": spatial_ratio_to_k[r],
        "temporal_K": temporal_ratio_to_k[r],
    }
    for r in matched_ratios
])

print("Spatial K values present in BOTH within and across:")
print(spatial_common_k)
print("\nTemporal K values present in BOTH within and across:")
print(temporal_common_k)
print("\nExact matched-ratio K pairs:")
display(matched_ratio_df)

matched_ratio_df.to_csv(TABLE_DIR / "exact_matched_ratio_K_pairs.csv", index=False)

In [ ]:
# ============================================================
# Load top-m decoding outputs
# ============================================================

def expected_topm_csv(base_dir, analysis, fit_scope):
    return Path(base_dir) / f"msaa_condrank_decoding_outputs_{analysis}_{fit_scope}" / "topm_summary.csv"

def standardize_topm_df(df):
    df = df.copy()

    if "analysis" not in df.columns and "analysis_type" in df.columns:
        df["analysis"] = df["analysis_type"]
    if "fit" not in df.columns and "fit_scope" in df.columns:
        df["fit"] = df["fit_scope"]

    if "accuracy" not in df.columns:
        if "mean" in df.columns:
            df["accuracy"] = df["mean"]
        elif "mean_accuracy" in df.columns:
            df["accuracy"] = df["mean_accuracy"]
        else:
            raise KeyError("Could not find accuracy column. Expected one of: accuracy, mean, mean_accuracy.")

    for c in ["analysis", "fit", "condition"]:
        df[c] = df[c].astype(str)

    df["K"] = df["K"].astype(int)
    df["top_m"] = df["top_m"].astype(int)
    df["D"] = df["analysis"].map(DIMENSIONS)
    df["component_ratio"] = df["K"] / df["D"]

    return df

def load_topm_outputs(base_dir=BASE_DIR):
    rows = []
    missing = []

    for analysis in ["spatial", "temporal"]:
        for fit_scope in ["across", "within"]:
            path = expected_topm_csv(base_dir, analysis, fit_scope)
            if not path.exists():
                missing.append(str(path))
                continue

            df = pd.read_csv(path)
            df["source_csv"] = str(path)

            if "analysis_type" not in df.columns:
                df["analysis_type"] = analysis
            if "fit_scope" not in df.columns:
                df["fit_scope"] = fit_scope

            rows.append(df)

    if not rows:
        raise FileNotFoundError(
            "No topm_summary.csv files found. Expected paths include:\n" + "\n".join(missing)
        )

    return standardize_topm_df(pd.concat(rows, ignore_index=True))

topm_df = load_topm_outputs(BASE_DIR)
print(topm_df.shape)
display(topm_df.head())
print(topm_df.groupby(["analysis", "fit", "condition"])[["K", "top_m"]].nunique())

In [ ]:
# ============================================================
# Restrict to exact matched ratios only
# ============================================================

matched_k_lookup = {
    "spatial": set(matched_ratio_df["spatial_K"].astype(int)),
    "temporal": set(matched_ratio_df["temporal_K"].astype(int)),
}

def restrict_to_exact_matched_ratios(df):
    keep = []
    for _, row in df.iterrows():
        analysis = row["analysis"]
        k = int(row["K"])
        keep.append(k in matched_k_lookup[analysis])
    return df.loc[keep].copy()

matched_topm_df = restrict_to_exact_matched_ratios(topm_df)

# Add a shared ratio column rounded only for display/grouping.
# The actual matching above used exact Fraction arithmetic from K_VALUES.
matched_topm_df["matched_ratio"] = matched_topm_df["component_ratio"].round(10)

print("Original rows:", len(topm_df))
print("Rows after exact matched-ratio filtering:", len(matched_topm_df))
print("Remaining K values by panel:")
display(matched_topm_df.groupby(["analysis", "fit"])["K"].unique().apply(lambda x: sorted(map(int, x))).reset_index())

matched_topm_df.to_csv(TABLE_DIR / "topm_summary_exact_matched_ratios_only.csv", index=False)

In [ ]:
# ============================================================
# Plotting helpers
# ============================================================

def savefig(fig, name):
    if not SAVE_FIGS:
        return None
    out = FIG_DIR / f"{name}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def get_panel_subset(df, analysis, fit, condition):
    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == fit)
        & (df["condition"] == condition)
        & (df["top_m"].isin(TOP_M_VALUES))
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows found for {analysis}, {fit}, {condition}")

    return sub

def build_exact_matched_heatmap(df, analysis, fit, condition):
    sub = get_panel_subset(df, analysis, fit, condition)

    ratio_values = matched_ratio_df["component_ratio"].to_numpy(float)
    heat = np.full((len(TOP_M_VALUES), len(ratio_values)), np.nan)

    if analysis == "spatial":
        k_col = "spatial_K"
    elif analysis == "temporal":
        k_col = "temporal_K"
    else:
        raise ValueError("analysis must be spatial or temporal")

    raw_k_values = matched_ratio_df[k_col].astype(int).tolist()

    for i, m in enumerate(TOP_M_VALUES):
        for j, k in enumerate(raw_k_values):
            vals = sub.loc[(sub["top_m"] == m) & (sub["K"] == k), "accuracy"]
            if len(vals):
                heat[i, j] = vals.mean()

    return heat, ratio_values, raw_k_values

def shared_color_limits(df, condition, panels):
    vals = []
    for analysis, fit in panels:
        sub = get_panel_subset(df, analysis, fit, condition)
        vals.extend(sub["accuracy"].dropna().tolist())
    return float(np.nanmin(vals)), float(np.nanmax(vals))

def plot_exact_matched_panel(df, analysis, fit, condition, ax=None, vmin=None, vmax=None, annotate_top_n=3):
    heat, ratio_values, raw_k_values = build_exact_matched_heatmap(df, analysis, fit, condition)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"{analysis.capitalize()} {fit}")
    ax.set_yticks(np.arange(len(TOP_M_VALUES)))
    ax.set_yticklabels(TOP_M_VALUES)
    ax.set_ylabel("top-m")

    ax.set_xticks(np.arange(len(ratio_values)))
    ax.set_xticklabels([f"{r:.2f}" for r in ratio_values], rotation=45, ha="right")
    ax.set_xlabel("Exact matched component ratio (K / D)")

    # Annotate top raw K values within each top-m row.
    for i, m in enumerate(TOP_M_VALUES):
        row = heat[i, :]
        valid = np.where(np.isfinite(row))[0]
        if len(valid) == 0:
            continue
        top_idx = valid[np.argsort(row[valid])[-annotate_top_n:]]
        for j in top_idx:
            ax.text(
                j, i, f"K={raw_k_values[j]}",
                ha="center", va="center",
                fontsize=6.5, color="white", fontweight="bold"
            )

    return fig, ax, im

def plot_condition_exact_matched_heatmaps(df, condition="intact", save=True):
    panels = [
        ("spatial", "across"),
        ("spatial", "within"),
        ("temporal", "across"),
        ("temporal", "within"),
    ]

    vmin, vmax = shared_color_limits(df, condition, panels)

    fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True, sharey=True)
    ims = []

    for ax, (analysis, fit) in zip(axes.flat, panels):
        _, _, im = plot_exact_matched_panel(
            df,
            analysis=analysis,
            fit=fit,
            condition=condition,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            annotate_top_n=3,
        )
        ims.append(im)

    fig.suptitle(f"Exact matched-ratio top-m decoding heatmaps: {condition}", fontsize=16)
    fig.tight_layout(rect=[0, 0, 0.92, 0.95])

    cax = fig.add_axes([0.93, 0.15, 0.015, 0.70])
    cbar = fig.colorbar(ims[0], cax=cax)
    cbar.set_label("Decoding accuracy")

    if save:
        savefig(fig, f"exact_matched_ratio_topm_decoding_heatmaps_{condition}")

    return fig

In [ ]:
# ============================================================
# Make exact matched-ratio heatmaps
# ============================================================

for condition in CONDITIONS:
    fig = plot_condition_exact_matched_heatmaps(matched_topm_df, condition=condition, save=True)
    plt.show()

In [ ]:
# ============================================================
# Build exact matched-ratio lookup table
# ============================================================

matched_rows = []

# Use ONLY K values shared between within/across
shared_temporal_K = sorted(
    set(K_VALUES[("temporal", "within")]).intersection(
        K_VALUES[("temporal", "across")]
    )
)

shared_spatial_K = sorted(
    set(K_VALUES[("spatial", "within")]).intersection(
        K_VALUES[("spatial", "across")]
    )
)

# Convert to ratio dictionaries
temporal_ratio_to_K = {
    round(k / 300, 4): k
    for k in shared_temporal_K
}

spatial_ratio_to_K = {
    round(k / 700, 4): k
    for k in shared_spatial_K
}

# Find exact overlapping ratios
matched_ratios = sorted(
    set(temporal_ratio_to_K.keys()).intersection(
        spatial_ratio_to_K.keys()
    )
)

for ratio in matched_ratios:
    matched_rows.append({
        "ratio": ratio,
        "spatial_K": spatial_ratio_to_K[ratio],
        "temporal_K": temporal_ratio_to_K[ratio],
    })

MATCHED_RATIO_TABLE = pd.DataFrame(matched_rows)

print(MATCHED_RATIO_TABLE)

display(MATCHED_RATIO_TABLE)

In [ ]:
# ============================================================
# Difference heatmaps: spatial minus temporal decoding
# ============================================================

def build_difference_df(df, fit="across", condition="intact"):
    """
    Computes spatial - temporal decoding accuracy at exact matched component ratios.

    Requires MATCHED_RATIO_TABLE from the notebook, with columns:
      ratio, spatial_K, temporal_K

    Returns one row per ratio x top_m.
    """
    rows = []

    for _, r in MATCHED_RATIO_TABLE.iterrows():
        ratio = float(r["ratio"])
        spatial_K = int(r["spatial_K"])
        temporal_K = int(r["temporal_K"])

        for top_m in TOP_M_VALUES:
            s = df[
                (df["analysis"] == "spatial")
                & (df["fit"] == fit)
                & (df["condition"] == condition)
                & (df["K"] == spatial_K)
                & (df["top_m"] == top_m)
            ]

            t = df[
                (df["analysis"] == "temporal")
                & (df["fit"] == fit)
                & (df["condition"] == condition)
                & (df["K"] == temporal_K)
                & (df["top_m"] == top_m)
            ]

            if len(s) == 0 or len(t) == 0:
                continue

            spatial_acc = float(s["accuracy"].mean())
            temporal_acc = float(t["accuracy"].mean())

            rows.append({
                "fit": fit,
                "condition": condition,
                "ratio": ratio,
                "spatial_K": spatial_K,
                "temporal_K": temporal_K,
                "top_m": int(top_m),
                "spatial_accuracy": spatial_acc,
                "temporal_accuracy": temporal_acc,
                "difference_spatial_minus_temporal": spatial_acc - temporal_acc,
            })

    return pd.DataFrame(rows)


def plot_difference_heatmap(diff_df, fit="across", condition="intact", ax=None):
    """
    Heatmap where values are spatial - temporal decoding accuracy.
    Positive values mean spatial AA decodes better.
    Negative values mean temporal AA decodes better.
    """
    sub = diff_df[
        (diff_df["fit"] == fit)
        & (diff_df["condition"] == condition)
    ].copy()

    if sub.empty:
        raise ValueError(f"No difference rows for fit={fit}, condition={condition}")

    ratios = sorted(sub["ratio"].unique())
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(ratios)), np.nan)

    for i, m in enumerate(top_ms):
        for j, ratio in enumerate(ratios):
            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["ratio"] == ratio),
                "difference_spatial_minus_temporal"
            ]
            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = np.nanmax(np.abs(heat))

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="coolwarm",
        vmin=-max_abs,
        vmax=max_abs,
    )

    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)
    ax.set_ylabel("top-m")

    ax.set_xticks(np.arange(len(ratios)))
    ax.set_xticklabels([f"{r:.2f}" for r in ratios], rotation=45, ha="right")
    ax.set_xlabel("Matched component ratio (K / D)")

    ax.set_title(f"Spatial minus temporal decoding: {fit}, {condition}")

    # Annotate raw matched K pairs
    for j, ratio in enumerate(ratios):
        row = sub[sub["ratio"] == ratio].iloc[0]
        ax.text(
            j,
            -0.85,
            f"S{int(row['spatial_K'])}\nT{int(row['temporal_K'])}",
            ha="center",
            va="top",
            fontsize=7,
            transform=ax.transData,
        )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Decoding difference: spatial - temporal")

    return fig, ax, im


# Build difference dataframe
all_diff_rows = []

for fit in ["across", "within"]:
    for condition in CONDITIONS:
        all_diff_rows.append(
            build_difference_df(topm_df, fit=fit, condition=condition)
        )

diff_df = pd.concat(all_diff_rows, ignore_index=True)

# Save table
diff_out = TABLE_DIR / "spatial_minus_temporal_exact_matched_ratio_differences.csv"
diff_df.to_csv(diff_out, index=False)
print("Saved:", diff_out)

display(diff_df.head())


# Plot one figure per condition: across and within side-by-side
for condition in CONDITIONS:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

    plot_difference_heatmap(diff_df, fit="across", condition=condition, ax=axes[0])
    plot_difference_heatmap(diff_df, fit="within", condition=condition, ax=axes[1])

    fig.suptitle(f"Spatial - temporal top-m decoding difference: {condition}", fontsize=15)
    fig.tight_layout()

    out = FIG_DIR / f"spatial_minus_temporal_difference_heatmap_{condition}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)

    plt.show()

In [ ]:
# ============================================================
# Difference heatmap with top-2 positive and negative annotations
# ============================================================

def plot_difference_heatmap(diff_df, fit="across", condition="intact", ax=None):

    sub = diff_df[
        (diff_df["fit"] == fit)
        & (diff_df["condition"] == condition)
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for fit={fit}, condition={condition}")

    ratios = sorted(sub["ratio"].unique())
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(ratios)), np.nan)

    for i, m in enumerate(top_ms):
        for j, ratio in enumerate(ratios):

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["ratio"] == ratio),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = np.nanmax(np.abs(heat))

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)
    ax.set_ylabel("top-m")

    ax.set_xticks(np.arange(len(ratios)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in ratios],
        rotation=45,
        ha="right"
    )

    ax.set_xlabel("Matched component ratio (K / D)")

    ax.set_title(
        f"Spatial minus temporal decoding\n{fit}, {condition}"
    )

    # --------------------------------------------------------
    # Annotate top 2 positive and top 2 negative cells
    # per top-m row
    # --------------------------------------------------------

    for i, m in enumerate(top_ms):

        row_vals = heat[i, :]
        valid = np.where(~np.isnan(row_vals))[0]

        if len(valid) == 0:
            continue

        # Top 2 spatial > temporal
        pos_idx = valid[np.argsort(row_vals[valid])[-2:]]

        # Top 2 temporal > spatial
        neg_idx = valid[np.argsort(row_vals[valid])[:2]]

        annotate_idx = np.unique(
            np.concatenate([pos_idx, neg_idx])
        )

        for j in annotate_idx:

            ratio = ratios[j]

            row = sub[
                (sub["ratio"] == ratio)
                & (sub["top_m"] == m)
            ].iloc[0]

            sK = int(row["spatial_K"])
            tK = int(row["temporal_K"])

            label = f"S{sK}\nT{tK}"

            val = heat[i, j]

            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=6,
                color="white" if abs(val) > max_abs * 0.35 else "black",
                fontweight="bold",
            )

    # --------------------------------------------------------
    # Colorbar
    # --------------------------------------------------------

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Spatial − temporal decoding")

    return fig, ax, im


# ============================================================
# Make plots
# ============================================================

for condition in CONDITIONS:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(15, 5),
        sharey=True
    )

    plot_difference_heatmap(
        diff_df,
        fit="across",
        condition=condition,
        ax=axes[0]
    )

    plot_difference_heatmap(
        diff_df,
        fit="within",
        condition=condition,
        ax=axes[1]
    )

    fig.suptitle(
        f"Spatial minus temporal top-m decoding difference: {condition}",
        fontsize=15
    )

    fig.tight_layout()

    out = FIG_DIR / f"spatial_minus_temporal_difference_heatmap_{condition}.{FIG_FORMAT}"

    fig.savefig(
        out,
        dpi=DPI,
        bbox_inches="tight"
    )

    print("Saved:", out)

    plt.show()

In [ ]:
# ============================================================
# Spatial-across vs temporal-across difference heatmaps
# using all exact matched ratio pairs
# ============================================================

def build_across_only_matched_ratio_table():
    """
    Exact ratio matches between spatial across and temporal across only.
    Does NOT require values to exist in within-condition fits.
    """
    spatial_K = K_VALUES[("spatial", "across")]
    temporal_K = K_VALUES[("temporal", "across")]

    spatial_ratio_to_K = {
        round(k / DIMENSIONS["spatial"], 4): k
        for k in spatial_K
    }

    temporal_ratio_to_K = {
        round(k / DIMENSIONS["temporal"], 4): k
        for k in temporal_K
    }

    matched_ratios = sorted(
        set(spatial_ratio_to_K.keys()).intersection(
            temporal_ratio_to_K.keys()
        )
    )

    rows = []
    for ratio in matched_ratios:
        rows.append({
            "ratio": ratio,
            "spatial_K": spatial_ratio_to_K[ratio],
            "temporal_K": temporal_ratio_to_K[ratio],
        })

    return pd.DataFrame(rows)


ACROSS_ONLY_MATCHED_RATIO_TABLE = build_across_only_matched_ratio_table()

display(ACROSS_ONLY_MATCHED_RATIO_TABLE)


def build_across_only_difference_df(df):
    """
    Computes spatial-across minus temporal-across decoding accuracy
    at exact matched component ratios.
    """
    rows = []

    for _, r in ACROSS_ONLY_MATCHED_RATIO_TABLE.iterrows():

        ratio = float(r["ratio"])
        spatial_K = int(r["spatial_K"])
        temporal_K = int(r["temporal_K"])

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:

                s = df[
                    (df["analysis"] == "spatial")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == spatial_K)
                    & (df["top_m"] == top_m)
                ]

                t = df[
                    (df["analysis"] == "temporal")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == temporal_K)
                    & (df["top_m"] == top_m)
                ]

                if len(s) == 0 or len(t) == 0:
                    continue

                spatial_acc = float(s["accuracy"].mean())
                temporal_acc = float(t["accuracy"].mean())

                rows.append({
                    "fit": "across",
                    "condition": condition,
                    "ratio": ratio,
                    "spatial_K": spatial_K,
                    "temporal_K": temporal_K,
                    "top_m": int(top_m),
                    "spatial_accuracy": spatial_acc,
                    "temporal_accuracy": temporal_acc,
                    "difference_spatial_minus_temporal": spatial_acc - temporal_acc,
                })

    return pd.DataFrame(rows)


across_diff_df = build_across_only_difference_df(topm_df)

out_csv = TABLE_DIR / "spatial_across_minus_temporal_across_all_matched_ratio_differences.csv"
across_diff_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

display(across_diff_df.head())


def plot_across_only_difference_heatmap(diff_df, condition="intact", ax=None):
    """
    Heatmap where values are spatial-across minus temporal-across decoding.
    Positive = spatial across better.
    Negative = temporal across better.
    """
    sub = diff_df[diff_df["condition"] == condition].copy()

    if sub.empty:
        raise ValueError(f"No rows for condition={condition}")

    ratios = sorted(sub["ratio"].unique())
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(ratios)), np.nan)

    for i, m in enumerate(top_ms):
        for j, ratio in enumerate(ratios):

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["ratio"] == ratio),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = np.nanmax(np.abs(heat))

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )

    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)
    ax.set_ylabel("top-m")

    ax.set_xticks(np.arange(len(ratios)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in ratios],
        rotation=45,
        ha="right"
    )
    ax.set_xlabel("Matched component ratio (K / D)")

    ax.set_title(f"Spatial across − temporal across decoding: {condition}")

    # Annotate top 2 positive and top 2 negative cells per top-m row
    for i, m in enumerate(top_ms):

        row_vals = heat[i, :]
        valid = np.where(~np.isnan(row_vals))[0]

        if len(valid) == 0:
            continue

        pos_idx = valid[np.argsort(row_vals[valid])[-2:]]
        neg_idx = valid[np.argsort(row_vals[valid])[:2]]

        annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

        for j in annotate_idx:

            ratio = ratios[j]

            row = sub[
                (sub["ratio"] == ratio)
                & (sub["top_m"] == m)
            ].iloc[0]

            sK = int(row["spatial_K"])
            tK = int(row["temporal_K"])
            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{sK}\nT{tK}",
                ha="center",
                va="center",
                fontsize=6,
                color="white" if abs(val) > max_abs * 0.35 else "black",
                fontweight="bold",
            )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Spatial across − temporal across decoding")

    return fig, ax, im


# ============================================================
# Plot one spatial-across minus temporal-across figure per condition
# ============================================================

for condition in CONDITIONS:

    fig, ax = plt.subplots(figsize=(13, 5))

    plot_across_only_difference_heatmap(
        across_diff_df,
        condition=condition,
        ax=ax
    )

    fig.tight_layout()

    out = FIG_DIR / f"spatial_across_minus_temporal_across_difference_heatmap_{condition}.{FIG_FORMAT}"

    fig.savefig(out, dpi=DPI, bbox_inches="tight")

    print("Saved:", out)

    plt.show()

In [ ]:
# ============================================================
# USER-DEFINED MATCHED K VALUES
# ============================================================

MATCHED_K_VALUES = {
    "spatial": [
        5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50,
        52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100,
        105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500,
        600, 700
    ],

    "temporal": [
        2, 3, 5, 6, 9, 11, 15, 17, 18, 18, 20, 20, 20, 23,
        23, 23, 23, 23, 23, 25, 27, 27, 27, 30, 35, 38, 45,
        45, 54, 60, 75, 81, 90, 90, 108, 129, 171, 214, 257, 300
    ]
}

MATCHED_RATIO_TABLE = pd.DataFrame({
    "spatial_K": MATCHED_K_VALUES["spatial"],
    "temporal_K": MATCHED_K_VALUES["temporal"],
})

MATCHED_RATIO_TABLE["spatial_ratio"] = (
    MATCHED_RATIO_TABLE["spatial_K"] / 700
)

MATCHED_RATIO_TABLE["temporal_ratio"] = (
    MATCHED_RATIO_TABLE["temporal_K"] / 300
)

MATCHED_RATIO_TABLE["matched_ratio"] = (
    MATCHED_RATIO_TABLE[["spatial_ratio", "temporal_ratio"]]
    .mean(axis=1)
)

MATCHED_RATIO_TABLE["ratio_difference"] = (
    np.abs(
        MATCHED_RATIO_TABLE["spatial_ratio"]
        - MATCHED_RATIO_TABLE["temporal_ratio"]
    )
)

display(MATCHED_RATIO_TABLE)

# ============================================================
# Restrict dataframe to matched K values only
# ============================================================

matched_k_lookup = {
    "spatial": set(MATCHED_K_VALUES["spatial"]),
    "temporal": set(MATCHED_K_VALUES["temporal"]),
}

matched_topm_df = topm_df[
    topm_df.apply(
        lambda r: int(r["K"]) in matched_k_lookup[r["analysis"]],
        axis=1
    )
].copy()

print("Original rows:", len(topm_df))
print("Matched rows:", len(matched_topm_df))

display(
    matched_topm_df.groupby(["analysis", "fit"])["K"]
    .unique()
    .apply(lambda x: sorted(map(int, x)))
    .reset_index()
)

# ============================================================
# Build heatmap using custom matched K ordering
# ============================================================

def build_custom_matched_heatmap(
    df,
    analysis,
    fit,
    condition
):

    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == fit)
        & (df["condition"] == condition)
        & (df["top_m"].isin(TOP_M_VALUES))
    ].copy()

    heat = np.full(
        (
            len(TOP_M_VALUES),
            len(MATCHED_RATIO_TABLE)
        ),
        np.nan
    )

    if analysis == "spatial":
        raw_k_values = MATCHED_RATIO_TABLE["spatial_K"].tolist()
    else:
        raw_k_values = MATCHED_RATIO_TABLE["temporal_K"].tolist()

    ratio_values = MATCHED_RATIO_TABLE["matched_ratio"].values

    for i, m in enumerate(TOP_M_VALUES):

        for j, k in enumerate(raw_k_values):

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["K"] == k),
                "accuracy"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, ratio_values, raw_k_values


# ============================================================
# Shared color scaling
# ============================================================

def shared_color_limits(df, condition, panels):

    vals = []

    for analysis, fit in panels:

        sub = df[
            (df["analysis"] == analysis)
            & (df["fit"] == fit)
            & (df["condition"] == condition)
        ]

        vals.extend(sub["accuracy"].dropna().tolist())

    return float(np.nanmin(vals)), float(np.nanmax(vals))


# ============================================================
# Plotting helper
# ============================================================

def plot_custom_matched_panel(
    df,
    analysis,
    fit,
    condition,
    ax=None,
    vmin=None,
    vmax=None,
    annotate_top_n=3,
):

    heat, ratio_values, raw_k_values = (
        build_custom_matched_heatmap(
            df,
            analysis,
            fit,
            condition
        )
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"{analysis.capitalize()} {fit}")

    ax.set_yticks(np.arange(len(TOP_M_VALUES)))
    ax.set_yticklabels(TOP_M_VALUES)
    ax.set_ylabel("top-m")

    ax.set_xticks(np.arange(len(ratio_values)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in ratio_values],
        rotation=45,
        ha="right"
    )

    ax.set_xlabel("Matched component ratio")

    # --------------------------------------------------------
    # Annotate top 3 decoding cells per row
    # --------------------------------------------------------

    for i, m in enumerate(TOP_M_VALUES):

        row = heat[i, :]
        valid = np.where(np.isfinite(row))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[
            np.argsort(row[valid])[-annotate_top_n:]
        ]

        for j in top_idx:

            ax.text(
                j,
                i,
                f"K={raw_k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    return fig, ax, im


# ============================================================
# Plot 2x2 matched-ratio heatmaps
# ============================================================

def plot_condition_custom_heatmaps(
    df,
    condition="intact",
    save=True
):

    panels = [
        ("spatial", "across"),
        ("spatial", "within"),
        ("temporal", "across"),
        ("temporal", "within"),
    ]

    vmin, vmax = shared_color_limits(
        df,
        condition,
        panels
    )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(16, 9),
        sharex=True,
        sharey=True
    )

    ims = []

    for ax, (analysis, fit) in zip(axes.flat, panels):

        _, _, im = plot_custom_matched_panel(
            df,
            analysis,
            fit,
            condition,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            annotate_top_n=3,
        )

        ims.append(im)

    fig.suptitle(
        f"Custom matched-ratio top-m decoding heatmaps: {condition}",
        fontsize=16
    )

    fig.tight_layout(rect=[0, 0, 0.92, 0.95])

    cax = fig.add_axes([0.93, 0.15, 0.015, 0.70])

    cbar = fig.colorbar(ims[0], cax=cax)

    cbar.set_label("Decoding accuracy")

    return fig


# ============================================================
# Generate figures
# ============================================================

for condition in CONDITIONS:

    fig = plot_condition_custom_heatmaps(
        matched_topm_df,
        condition=condition,
        save=False
    )

    plt.show()

In [ ]:
# ============================================================
# Difference heatmaps:
# 1) Across: custom spatial/temporal matched K pairs
# 2) Within: separate spatial and temporal panels using all available K values
#    with no missing gaps
# ============================================================

MATCHED_K_VALUES = {
    "spatial": [
        5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50,
        52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100,
        105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500,
        600, 700
    ],

    "temporal": [
        2, 3, 5, 6, 9, 11, 15, 17, 18, 18, 20, 20, 20, 23,
        23, 23, 23, 23, 23, 25, 27, 27, 27, 30, 35, 38, 45,
        45, 54, 60, 75, 81, 90, 90, 108, 129, 171, 214, 257,
        300
    ]
}

CUSTOM_MATCHED_RATIO_TABLE = pd.DataFrame({
    "spatial_K": MATCHED_K_VALUES["spatial"],
    "temporal_K": MATCHED_K_VALUES["temporal"],
})

CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"] = CUSTOM_MATCHED_RATIO_TABLE["spatial_K"] / DIMENSIONS["spatial"]
CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"] = CUSTOM_MATCHED_RATIO_TABLE["temporal_K"] / DIMENSIONS["temporal"]
CUSTOM_MATCHED_RATIO_TABLE["matched_ratio"] = CUSTOM_MATCHED_RATIO_TABLE[["spatial_ratio", "temporal_ratio"]].mean(axis=1)
CUSTOM_MATCHED_RATIO_TABLE["ratio_difference"] = np.abs(
    CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"] - CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"]
)

display(CUSTOM_MATCHED_RATIO_TABLE)


# ============================================================
# Across: spatial-across minus temporal-across using custom pairs
# ============================================================

def build_custom_across_difference_df(df):
    rows = []

    for _, r in CUSTOM_MATCHED_RATIO_TABLE.iterrows():
        spatial_K = int(r["spatial_K"])
        temporal_K = int(r["temporal_K"])

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:

                s = df[
                    (df["analysis"] == "spatial")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == spatial_K)
                    & (df["top_m"] == top_m)
                ]

                t = df[
                    (df["analysis"] == "temporal")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == temporal_K)
                    & (df["top_m"] == top_m)
                ]

                if len(s) == 0 or len(t) == 0:
                    continue

                spatial_acc = float(s["accuracy"].mean())
                temporal_acc = float(t["accuracy"].mean())

                rows.append({
                    "condition": condition,
                    "top_m": int(top_m),
                    "spatial_K": spatial_K,
                    "temporal_K": temporal_K,
                    "spatial_ratio": float(r["spatial_ratio"]),
                    "temporal_ratio": float(r["temporal_ratio"]),
                    "matched_ratio": float(r["matched_ratio"]),
                    "ratio_difference": float(r["ratio_difference"]),
                    "spatial_accuracy": spatial_acc,
                    "temporal_accuracy": temporal_acc,
                    "difference_spatial_minus_temporal": spatial_acc - temporal_acc,
                })

    return pd.DataFrame(rows)


custom_across_diff_df = build_custom_across_difference_df(topm_df)

out_csv = TABLE_DIR / "custom_spatial_across_minus_temporal_across_differences.csv"
custom_across_diff_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

display(custom_across_diff_df.head())


def plot_custom_across_difference_heatmap(diff_df, condition="intact", ax=None):
    sub = diff_df[diff_df["condition"] == condition].copy()

    if sub.empty:
        raise ValueError(f"No rows for condition={condition}")

    # Preserve user-defined matched order
    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(x_table)), np.nan)

    for i, m in enumerate(top_ms):
        for j, r in x_table.iterrows():
            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["spatial_K"] == int(r["spatial_K"]))
                & (sub["temporal_K"] == int(r["temporal_K"])),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = np.nanmax(np.abs(heat))

    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )

    ax.set_title(f"Across: spatial − temporal ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    ax.set_xticks(np.arange(len(x_table)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in x_table["matched_ratio"]],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Matched component ratio")

    # Annotate top 2 positive and top 2 negative per top-m row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        pos_idx = valid[np.argsort(row_vals[valid])[-2:]]
        neg_idx = valid[np.argsort(row_vals[valid])[:2]]
        annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

        for j in annotate_idx:
            spatial_K = int(x_table.iloc[j]["spatial_K"])
            temporal_K = int(x_table.iloc[j]["temporal_K"])
            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{spatial_K}\nT{temporal_K}",
                ha="center",
                va="center",
                fontsize=5.5,
                color="white" if abs(val) > max_abs * 0.35 else "black",
                fontweight="bold",
            )

    return fig, ax, im


# ============================================================
# Within: plot all available data separately, no matched axis required
# ============================================================

def build_single_analysis_within_heatmap(df, analysis, condition):
    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == "within")
        & (df["condition"] == condition)
        & (df["top_m"].isin(TOP_M_VALUES))
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}, within, condition={condition}")

    k_values = sorted(sub["K"].unique())
    ratios = [k / DIMENSIONS[analysis] for k in k_values]

    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(k_values)), np.nan)

    for i, m in enumerate(top_ms):
        for j, k in enumerate(k_values):
            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["K"] == k),
                "accuracy"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, k_values, ratios, top_ms


def plot_single_analysis_within_heatmap(df, analysis, condition, ax=None, vmin=None, vmax=None):
    heat, k_values, ratios, top_ms = build_single_analysis_within_heatmap(
        df,
        analysis=analysis,
        condition=condition,
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"Within: {analysis} ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    tick_idx = np.linspace(0, len(k_values) - 1, min(10, len(k_values)), dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        [f"K={k_values[i]}\n{ratios[i]:.2f}" for i in tick_idx],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Available K values (component ratio)")

    # annotate top 3 per top-m row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row_vals[valid])[-3:]]

        for j in top_idx:
            ax.text(
                j,
                i,
                f"K={k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    return fig, ax, im


# ============================================================
# Generate combined figure per condition
# Top panel: across difference
# Bottom panels: within spatial and temporal separately
# ============================================================

for condition in CONDITIONS:

    fig = plt.figure(figsize=(18, 11))

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[1.0, 1.0],
        width_ratios=[1, 1]
    )

    ax_across = fig.add_subplot(gs[0, :])
    ax_spatial_within = fig.add_subplot(gs[1, 0])
    ax_temporal_within = fig.add_subplot(gs[1, 1])

    _, _, im_diff = plot_custom_across_difference_heatmap(
        custom_across_diff_df,
        condition=condition,
        ax=ax_across,
    )

    # shared scale for within spatial/temporal panels within condition
    within_vals = topm_df[
        (topm_df["fit"] == "within")
        & (topm_df["condition"] == condition)
    ]["accuracy"].dropna()

    vmin = float(within_vals.min())
    vmax = float(within_vals.max())

    _, _, im_s = plot_single_analysis_within_heatmap(
        topm_df,
        analysis="spatial",
        condition=condition,
        ax=ax_spatial_within,
        vmin=vmin,
        vmax=vmax,
    )

    _, _, im_t = plot_single_analysis_within_heatmap(
        topm_df,
        analysis="temporal",
        condition=condition,
        ax=ax_temporal_within,
        vmin=vmin,
        vmax=vmax,
    )

    cbar1 = fig.colorbar(im_diff, ax=ax_across, fraction=0.025, pad=0.02)
    cbar1.set_label("Across decoding difference: spatial − temporal")

    cbar2 = fig.colorbar(
        im_s,
        ax=[ax_spatial_within, ax_temporal_within],
        fraction=0.025,
        pad=0.02,
    )
    cbar2.set_label("Within decoding accuracy")

    fig.suptitle(
        f"Top-m decoding structure: across difference and within analyses ({condition})",
        fontsize=16,
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out = FIG_DIR / f"custom_difference_and_within_heatmaps_{condition}.{FIG_FORMAT}"

    fig.savefig(out, dpi=DPI, bbox_inches="tight")

    print("Saved:", out)

    plt.show()

In [ ]:
# ============================================================
# Difference heatmaps with m > K masked
# 1) Across: custom spatial/temporal matched K pairs
# 2) Within: all available K values, separate spatial/temporal axes
# ============================================================

MATCHED_K_VALUES = {
    "spatial": [
        5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50,
        52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100,
        105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500,
        600, 700
    ],
    "temporal": [
        2, 3, 5, 6, 9, 11, 15, 17, 18, 18, 20, 20, 20, 23,
        23, 23, 23, 23, 23, 25, 27, 27, 27, 30, 35, 38, 45,
        45, 54, 60, 75, 81, 90, 90, 108, 129, 171, 214, 257,
        300
    ]
}

CUSTOM_MATCHED_RATIO_TABLE = pd.DataFrame({
    "spatial_K": MATCHED_K_VALUES["spatial"],
    "temporal_K": MATCHED_K_VALUES["temporal"],
})

CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"] = CUSTOM_MATCHED_RATIO_TABLE["spatial_K"] / DIMENSIONS["spatial"]
CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"] = CUSTOM_MATCHED_RATIO_TABLE["temporal_K"] / DIMENSIONS["temporal"]
CUSTOM_MATCHED_RATIO_TABLE["matched_ratio"] = CUSTOM_MATCHED_RATIO_TABLE[["spatial_ratio", "temporal_ratio"]].mean(axis=1)
CUSTOM_MATCHED_RATIO_TABLE["ratio_difference"] = np.abs(
    CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"] - CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"]
)

display(CUSTOM_MATCHED_RATIO_TABLE)


def build_custom_across_difference_df(df):
    rows = []

    for _, r in CUSTOM_MATCHED_RATIO_TABLE.iterrows():

        spatial_K = int(r["spatial_K"])
        temporal_K = int(r["temporal_K"])

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:

                # Mask saturated cases:
                # if m > K, top-m is identical to full K-component reconstruction
                if top_m > spatial_K or top_m > temporal_K:
                    continue

                s = df[
                    (df["analysis"] == "spatial")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == spatial_K)
                    & (df["top_m"] == top_m)
                ]

                t = df[
                    (df["analysis"] == "temporal")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == temporal_K)
                    & (df["top_m"] == top_m)
                ]

                if len(s) == 0 or len(t) == 0:
                    continue

                spatial_acc = float(s["accuracy"].mean())
                temporal_acc = float(t["accuracy"].mean())

                rows.append({
                    "condition": condition,
                    "top_m": int(top_m),
                    "spatial_K": spatial_K,
                    "temporal_K": temporal_K,
                    "spatial_ratio": float(r["spatial_ratio"]),
                    "temporal_ratio": float(r["temporal_ratio"]),
                    "matched_ratio": float(r["matched_ratio"]),
                    "ratio_difference": float(r["ratio_difference"]),
                    "spatial_accuracy": spatial_acc,
                    "temporal_accuracy": temporal_acc,
                    "difference_spatial_minus_temporal": spatial_acc - temporal_acc,
                })

    return pd.DataFrame(rows)


custom_across_diff_df = build_custom_across_difference_df(topm_df)

out_csv = TABLE_DIR / "custom_spatial_across_minus_temporal_across_differences_m_gt_K_masked.csv"
custom_across_diff_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


def plot_custom_across_difference_heatmap(diff_df, condition="intact", ax=None):
    sub = diff_df[diff_df["condition"] == condition].copy()

    if sub.empty:
        raise ValueError(f"No rows for condition={condition}")

    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(x_table)), np.nan)

    for i, m in enumerate(top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            if m > spatial_K or m > temporal_K:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["spatial_K"] == spatial_K)
                & (sub["temporal_K"] == temporal_K),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = np.nanmax(np.abs(heat))

    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )

    ax.set_title(f"Across: spatial − temporal ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    ax.set_xticks(np.arange(len(x_table)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in x_table["matched_ratio"]],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Matched component ratio")

    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        pos_idx = valid[np.argsort(row_vals[valid])[-2:]]
        neg_idx = valid[np.argsort(row_vals[valid])[:2]]
        annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

        for j in annotate_idx:
            spatial_K = int(x_table.iloc[j]["spatial_K"])
            temporal_K = int(x_table.iloc[j]["temporal_K"])
            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{spatial_K}\nT{temporal_K}",
                ha="center",
                va="center",
                fontsize=5.5,
                color="white" if abs(val) > max_abs * 0.35 else "black",
                fontweight="bold",
            )

    return fig, ax, im


def build_single_analysis_within_heatmap(df, analysis, condition):
    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == "within")
        & (df["condition"] == condition)
        & (df["top_m"].isin(TOP_M_VALUES))
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}, within, condition={condition}")

    k_values = sorted(sub["K"].unique())
    ratios = [k / DIMENSIONS[analysis] for k in k_values]
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(k_values)), np.nan)

    for i, m in enumerate(top_ms):
        for j, k in enumerate(k_values):

            # Mask saturated cases where top-m exceeds available components
            if m > k:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["K"] == k),
                "accuracy"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, k_values, ratios, top_ms


def plot_single_analysis_within_heatmap(df, analysis, condition, ax=None, vmin=None, vmax=None):
    heat, k_values, ratios, top_ms = build_single_analysis_within_heatmap(
        df,
        analysis=analysis,
        condition=condition,
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"Within: {analysis} ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    tick_idx = np.linspace(0, len(k_values) - 1, min(10, len(k_values)), dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        [f"K={k_values[i]}\n{ratios[i]:.2f}" for i in tick_idx],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Available K values (component ratio)")

    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row_vals[valid])[-3:]]

        for j in top_idx:
            ax.text(
                j,
                i,
                f"K={k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    return fig, ax, im


for condition in CONDITIONS:

    fig = plt.figure(figsize=(18, 11))

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[1.0, 1.0],
        width_ratios=[1, 1]
    )

    ax_across = fig.add_subplot(gs[0, :])
    ax_spatial_within = fig.add_subplot(gs[1, 0])
    ax_temporal_within = fig.add_subplot(gs[1, 1])

    _, _, im_diff = plot_custom_across_difference_heatmap(
        custom_across_diff_df,
        condition=condition,
        ax=ax_across,
    )

    within_vals = topm_df[
        (topm_df["fit"] == "within")
        & (topm_df["condition"] == condition)
    ]["accuracy"].dropna()

    vmin = float(within_vals.min())
    vmax = float(within_vals.max())

    _, _, im_s = plot_single_analysis_within_heatmap(
        topm_df,
        analysis="spatial",
        condition=condition,
        ax=ax_spatial_within,
        vmin=vmin,
        vmax=vmax,
    )

    _, _, im_t = plot_single_analysis_within_heatmap(
        topm_df,
        analysis="temporal",
        condition=condition,
        ax=ax_temporal_within,
        vmin=vmin,
        vmax=vmax,
    )

    cbar1 = fig.colorbar(im_diff, ax=ax_across, fraction=0.025, pad=0.02)
    cbar1.set_label("Across decoding difference: spatial − temporal")

    cbar2 = fig.colorbar(
        im_s,
        ax=[ax_spatial_within, ax_temporal_within],
        fraction=0.025,
        pad=0.02,
    )
    cbar2.set_label("Within decoding accuracy")

    fig.suptitle(
        f"Top-m decoding structure with m > K masked ({condition})",
        fontsize=16,
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out = FIG_DIR / f"custom_difference_and_within_heatmaps_m_gt_K_masked_{condition}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")

    print("Saved:", out)
    plt.show()

In [ ]:
# ============================================================
# Combined heatmaps with m > K masked
#
# Top row:
#   Across difference heatmap: spatial across − temporal across
#
# Middle row:
#   Across spatial and temporal decoding heatmaps
#
# Bottom row:
#   Within spatial and temporal decoding heatmaps
# ============================================================

MATCHED_K_VALUES = {
    "spatial": [
        5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50,
        52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100,
        105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500,
        600, 700
    ],
    "temporal": [
        2, 3, 5, 6, 9, 11, 15, 17, 18, 18, 20, 20, 20, 23,
        23, 23, 23, 23, 23, 25, 27, 27, 27, 30, 35, 38, 45,
        45, 54, 60, 75, 81, 90, 90, 108, 129, 171, 214, 257,
        300
    ]
}

CUSTOM_MATCHED_RATIO_TABLE = pd.DataFrame({
    "spatial_K": MATCHED_K_VALUES["spatial"],
    "temporal_K": MATCHED_K_VALUES["temporal"],
})

CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"] = CUSTOM_MATCHED_RATIO_TABLE["spatial_K"] / DIMENSIONS["spatial"]
CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"] = CUSTOM_MATCHED_RATIO_TABLE["temporal_K"] / DIMENSIONS["temporal"]
CUSTOM_MATCHED_RATIO_TABLE["matched_ratio"] = CUSTOM_MATCHED_RATIO_TABLE[["spatial_ratio", "temporal_ratio"]].mean(axis=1)
CUSTOM_MATCHED_RATIO_TABLE["ratio_difference"] = np.abs(
    CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"] - CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"]
)

display(CUSTOM_MATCHED_RATIO_TABLE)


# ============================================================
# Build across difference dataframe
# ============================================================

def build_custom_across_difference_df(df):
    rows = []

    for _, r in CUSTOM_MATCHED_RATIO_TABLE.iterrows():

        spatial_K = int(r["spatial_K"])
        temporal_K = int(r["temporal_K"])

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:

                # Mask saturated cases
                if top_m > spatial_K or top_m > temporal_K:
                    continue

                s = df[
                    (df["analysis"] == "spatial")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == spatial_K)
                    & (df["top_m"] == top_m)
                ]

                t = df[
                    (df["analysis"] == "temporal")
                    & (df["fit"] == "across")
                    & (df["condition"] == condition)
                    & (df["K"] == temporal_K)
                    & (df["top_m"] == top_m)
                ]

                if len(s) == 0 or len(t) == 0:
                    continue

                spatial_acc = float(s["accuracy"].mean())
                temporal_acc = float(t["accuracy"].mean())

                rows.append({
                    "condition": condition,
                    "top_m": int(top_m),
                    "spatial_K": spatial_K,
                    "temporal_K": temporal_K,
                    "spatial_ratio": float(r["spatial_ratio"]),
                    "temporal_ratio": float(r["temporal_ratio"]),
                    "matched_ratio": float(r["matched_ratio"]),
                    "ratio_difference": float(r["ratio_difference"]),
                    "spatial_accuracy": spatial_acc,
                    "temporal_accuracy": temporal_acc,
                    "difference_spatial_minus_temporal": spatial_acc - temporal_acc,
                })

    return pd.DataFrame(rows)


custom_across_diff_df = build_custom_across_difference_df(topm_df)

out_csv = TABLE_DIR / "custom_spatial_across_minus_temporal_across_differences_m_gt_K_masked.csv"
custom_across_diff_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


# ============================================================
# Across difference heatmap
# ============================================================

def plot_custom_across_difference_heatmap(diff_df, condition="intact", ax=None):
    sub = diff_df[diff_df["condition"] == condition].copy()

    if sub.empty:
        raise ValueError(f"No rows for condition={condition}")

    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(x_table)), np.nan)

    for i, m in enumerate(top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            if m > spatial_K or m > temporal_K:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["spatial_K"] == spatial_K)
                & (sub["temporal_K"] == temporal_K),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = np.nanmax(np.abs(heat))

    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )

    ax.set_title(f"Across difference: spatial − temporal ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    ax.set_xticks(np.arange(len(x_table)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in x_table["matched_ratio"]],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Matched component ratio")

    # annotate top 2 positive and top 2 negative per top-m row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        pos_idx = valid[np.argsort(row_vals[valid])[-5:]]
        neg_idx = valid[np.argsort(row_vals[valid])[:5]]
        annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

        for j in annotate_idx:
            spatial_K = int(x_table.iloc[j]["spatial_K"])
            temporal_K = int(x_table.iloc[j]["temporal_K"])
            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{spatial_K}\nT{temporal_K}",
                ha="center",
                va="center",
                fontsize=5.5,
                color="white" if abs(val) > max_abs * 0.35 else "black",
                fontweight="bold",
            )

    return fig, ax, im


# ============================================================
# Accuracy heatmaps for custom across matched K values
# ============================================================

def build_custom_across_accuracy_heatmap(df, analysis, condition):
    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == "across")
        & (df["condition"] == condition)
        & (df["top_m"].isin(TOP_M_VALUES))
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}, across, condition={condition}")

    if analysis == "spatial":
        k_values = CUSTOM_MATCHED_RATIO_TABLE["spatial_K"].astype(int).tolist()
        ratios = CUSTOM_MATCHED_RATIO_TABLE["spatial_ratio"].tolist()
    elif analysis == "temporal":
        k_values = CUSTOM_MATCHED_RATIO_TABLE["temporal_K"].astype(int).tolist()
        ratios = CUSTOM_MATCHED_RATIO_TABLE["temporal_ratio"].tolist()
    else:
        raise ValueError("analysis must be 'spatial' or 'temporal'")

    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(k_values)), np.nan)

    for i, m in enumerate(top_ms):
        for j, k in enumerate(k_values):

            if m > k:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["K"] == k),
                "accuracy"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, k_values, ratios, top_ms


def plot_custom_across_accuracy_heatmap(df, analysis, condition, ax=None, vmin=None, vmax=None):
    heat, k_values, ratios, top_ms = build_custom_across_accuracy_heatmap(
        df,
        analysis=analysis,
        condition=condition,
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"Across: {analysis} ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    tick_idx = np.linspace(0, len(k_values) - 1, min(12, len(k_values)), dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        [f"K={k_values[i]}\n{ratios[i]:.2f}" for i in tick_idx],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Matched K values (component ratio)")

    # annotate top 3 per top-m row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row_vals[valid])[-3:]]

        for j in top_idx:
            ax.text(
                j,
                i,
                f"K={k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    return fig, ax, im


# ============================================================
# Within heatmaps: all available K values, no gaps
# ============================================================

def build_single_analysis_within_heatmap(df, analysis, condition):
    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == "within")
        & (df["condition"] == condition)
        & (df["top_m"].isin(TOP_M_VALUES))
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}, within, condition={condition}")

    k_values = sorted(sub["K"].unique())
    ratios = [k / DIMENSIONS[analysis] for k in k_values]
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(k_values)), np.nan)

    for i, m in enumerate(top_ms):
        for j, k in enumerate(k_values):

            if m > k:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["K"] == k),
                "accuracy"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, k_values, ratios, top_ms


def plot_single_analysis_within_heatmap(df, analysis, condition, ax=None, vmin=None, vmax=None):
    heat, k_values, ratios, top_ms = build_single_analysis_within_heatmap(
        df,
        analysis=analysis,
        condition=condition,
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))
    else:
        fig = ax.figure

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"Within: {analysis} ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    tick_idx = np.linspace(0, len(k_values) - 1, min(10, len(k_values)), dtype=int)
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        [f"K={k_values[i]}\n{ratios[i]:.2f}" for i in tick_idx],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Available K values (component ratio)")

    # annotate top 3 per top-m row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row_vals[valid])[-3:]]

        for j in top_idx:
            ax.text(
                j,
                i,
                f"K={k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    return fig, ax, im


# ============================================================
# Generate combined figure per condition
# ============================================================

for condition in CONDITIONS:

    fig = plt.figure(figsize=(20, 15))

    gs = fig.add_gridspec(
        3,
        2,
        height_ratios=[0.9, 1.0, 1.0],
        width_ratios=[1, 1],
    )

    ax_diff = fig.add_subplot(gs[0, :])

    ax_across_spatial = fig.add_subplot(gs[1, 0])
    ax_across_temporal = fig.add_subplot(gs[1, 1])

    ax_within_spatial = fig.add_subplot(gs[2, 0])
    ax_within_temporal = fig.add_subplot(gs[2, 1])

    # difference heatmap
    _, _, im_diff = plot_custom_across_difference_heatmap(
        custom_across_diff_df,
        condition=condition,
        ax=ax_diff,
    )

    # shared accuracy scale across across + within accuracy panels
    accuracy_vals = topm_df[
        (topm_df["condition"] == condition)
        & (topm_df["fit"].isin(["across", "within"]))
    ]["accuracy"].dropna()

    vmin = float(accuracy_vals.min())
    vmax = float(accuracy_vals.max())

    # across accuracy heatmaps
    _, _, im_a_s = plot_custom_across_accuracy_heatmap(
        topm_df,
        analysis="spatial",
        condition=condition,
        ax=ax_across_spatial,
        vmin=vmin,
        vmax=vmax,
    )

    _, _, im_a_t = plot_custom_across_accuracy_heatmap(
        topm_df,
        analysis="temporal",
        condition=condition,
        ax=ax_across_temporal,
        vmin=vmin,
        vmax=vmax,
    )

    # within accuracy heatmaps
    _, _, im_w_s = plot_single_analysis_within_heatmap(
        topm_df,
        analysis="spatial",
        condition=condition,
        ax=ax_within_spatial,
        vmin=vmin,
        vmax=vmax,
    )

    _, _, im_w_t = plot_single_analysis_within_heatmap(
        topm_df,
        analysis="temporal",
        condition=condition,
        ax=ax_within_temporal,
        vmin=vmin,
        vmax=vmax,
    )

    cbar1 = fig.colorbar(im_diff, ax=ax_diff, fraction=0.025, pad=0.02)
    cbar1.set_label("Across decoding difference: spatial − temporal")

    cbar2 = fig.colorbar(
        im_a_s,
        ax=[
            ax_across_spatial,
            ax_across_temporal,
            ax_within_spatial,
            ax_within_temporal,
        ],
        fraction=0.025,
        pad=0.02,
    )
    cbar2.set_label("Decoding accuracy")

    fig.suptitle(
        f"Top-m decoding heatmaps with m > K masked ({condition})",
        fontsize=18,
    )

    # fig.tight_layout(rect=[0, 0, 1, 0.97])

    out = FIG_DIR / f"combined_difference_across_within_heatmaps_m_gt_K_masked_{condition}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")

    print("Saved:", out)
    plt.show()

In [ ]:
# ============================================================
# Save each heatmap separately for Illustrator
# - Across difference heatmap
# - Across spatial accuracy heatmap, same x-axis/order as difference
# - Across temporal accuracy heatmap, same x-axis/order as difference
# - Within spatial accuracy heatmap, all available K values
# - Within temporal accuracy heatmap, all available K values
# - One shared decoding colorbar
# ============================================================

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

SEPARATE_FIG_DIR = FIG_DIR / "separate_heatmaps_for_illustrator"
SEPARATE_FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Shared decoding color scale across all accuracy heatmaps
# ------------------------------------------------------------

global_vmin = float(topm_df["accuracy"].dropna().min())
global_vmax = float(topm_df["accuracy"].dropna().max())

print("Shared decoding color limits:", global_vmin, global_vmax)


# ------------------------------------------------------------
# Helper: save standalone colorbar
# ------------------------------------------------------------

def save_standalone_colorbar(
    vmin,
    vmax,
    cmap="viridis",
    label="Decoding accuracy",
    filename="shared_decoding_colorbar.pdf",
):
    fig, ax = plt.subplots(figsize=(1.2, 5))
    fig.subplots_adjust(left=0.45, right=0.75)

    norm = Normalize(vmin=vmin, vmax=vmax)
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=ax)
    cbar.set_label(label)

    out = SEPARATE_FIG_DIR / filename
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out)


save_standalone_colorbar(
    global_vmin,
    global_vmax,
    cmap="viridis",
    label="Decoding accuracy",
    filename=f"shared_decoding_accuracy_colorbar.{FIG_FORMAT}",
)


# ------------------------------------------------------------
# Helper: save standalone difference colorbar per condition
# ------------------------------------------------------------

def save_difference_colorbar(max_abs, condition):
    fig, ax = plt.subplots(figsize=(1.2, 5))
    fig.subplots_adjust(left=0.45, right=0.75)

    norm = Normalize(vmin=-max_abs, vmax=max_abs)
    sm = ScalarMappable(norm=norm, cmap="RdBu_r")
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=ax)
    cbar.set_label("Spatial − temporal decoding")

    out = SEPARATE_FIG_DIR / f"{condition}_difference_colorbar.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out)


# ------------------------------------------------------------
# Difference heatmap, no colorbar
# ------------------------------------------------------------

def save_across_difference_heatmap_separate(diff_df, condition):

    fig, ax = plt.subplots(figsize=(16, 4.8))

    sub = diff_df[diff_df["condition"] == condition].copy()
    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(x_table)), np.nan)

    for i, m in enumerate(top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            if m > spatial_K or m > temporal_K:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["spatial_K"] == spatial_K)
                & (sub["temporal_K"] == temporal_K),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    max_abs = float(np.nanmax(np.abs(heat)))

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )

    ax.set_title(f"Across difference: spatial − temporal ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    ax.set_xticks(np.arange(len(x_table)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in x_table["matched_ratio"]],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Matched component ratio")

    # annotate top 5 positive and negative per top-m row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        pos_idx = valid[np.argsort(row_vals[valid])[-5:]]
        neg_idx = valid[np.argsort(row_vals[valid])[:5]]
        annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

        for j in annotate_idx:
            spatial_K = int(x_table.iloc[j]["spatial_K"])
            temporal_K = int(x_table.iloc[j]["temporal_K"])
            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{spatial_K}\nT{temporal_K}",
                ha="center",
                va="center",
                fontsize=5.5,
                color="white" if abs(val) > max_abs * 0.35 else "black",
                fontweight="bold",
            )

    fig.tight_layout()

    out = SEPARATE_FIG_DIR / f"{condition}_across_difference_spatial_minus_temporal.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out)

    save_difference_colorbar(max_abs, condition)


# ------------------------------------------------------------
# Across accuracy heatmap with SAME x-axis/order as difference
# ------------------------------------------------------------

def save_across_accuracy_heatmap_separate(df, analysis, condition):

    heat, k_values, ratios, top_ms = build_custom_across_accuracy_heatmap(
        df,
        analysis=analysis,
        condition=condition,
    )

    fig, ax = plt.subplots(figsize=(16, 4.8))

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=global_vmin,
        vmax=global_vmax,
    )

    ax.set_title(f"Across: {analysis} ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    # IMPORTANT: same x-axis/order as difference heatmap
    ax.set_xticks(np.arange(len(CUSTOM_MATCHED_RATIO_TABLE)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in CUSTOM_MATCHED_RATIO_TABLE["matched_ratio"]],
        rotation=45,
        ha="right",
    )
    ax.set_xlabel("Matched component ratio")

    # annotate top 3 per row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row_vals[valid])[-3:]]

        for j in top_idx:
            ax.text(
                j,
                i,
                f"K={k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    fig.tight_layout()

    out = SEPARATE_FIG_DIR / f"{condition}_across_{analysis}_accuracy.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out)


# ------------------------------------------------------------
# Within accuracy heatmap, all available K values
# ------------------------------------------------------------

def save_within_accuracy_heatmap_separate(df, analysis, condition):

    heat, k_values, ratios, top_ms = build_single_analysis_within_heatmap(
        df,
        analysis=analysis,
        condition=condition,
    )

    fig, ax = plt.subplots(figsize=(14, 4.8))

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
        vmin=global_vmin,
        vmax=global_vmax,
    )

    ax.set_title(f"Within: {analysis} ({condition})")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms)

    ax.set_xticks(np.arange(len(k_values)))
    ax.set_xticklabels(
        [f"K={k}\n{k / DIMENSIONS[analysis]:.2f}" for k in k_values],
        rotation=45,
        ha="right",
        fontsize=7,
    )
    ax.set_xlabel("Available K values (component ratio)")

    # annotate top 3 per row
    for i, m in enumerate(top_ms):
        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row_vals[valid])[-3:]]

        for j in top_idx:
            ax.text(
                j,
                i,
                f"K={k_values[j]}",
                ha="center",
                va="center",
                fontsize=6,
                color="white",
                fontweight="bold",
            )

    fig.tight_layout()

    out = SEPARATE_FIG_DIR / f"{condition}_within_{analysis}_accuracy_all_available_K.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", out)


# ------------------------------------------------------------
# Save everything separately
# ------------------------------------------------------------

for condition in CONDITIONS:

    save_across_difference_heatmap_separate(
        custom_across_diff_df,
        condition=condition,
    )

    save_across_accuracy_heatmap_separate(
        topm_df,
        analysis="spatial",
        condition=condition,
    )

    save_across_accuracy_heatmap_separate(
        topm_df,
        analysis="temporal",
        condition=condition,
    )

    save_within_accuracy_heatmap_separate(
        topm_df,
        analysis="spatial",
        condition=condition,
    )

    save_within_accuracy_heatmap_separate(
        topm_df,
        analysis="temporal",
        condition=condition,
    )

In [ ]:
# ============================================================
# Remake raw full-reconstruction decoding figure
# Same style as full_recon_K_across.pdf
#
# Uses full_summary.csv from:
#   msaa_condrank_decoding_outputs_spatial_across/full_summary.csv
#   msaa_condrank_decoding_outputs_temporal_across/full_summary.csv
#   msaa_condrank_decoding_outputs_spatial_within/full_summary.csv
#   msaa_condrank_decoding_outputs_temporal_within/full_summary.csv
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

BASE_DIR = Path(".")
FIG_DIR = Path("figures") / "full_reconstruction_decoding"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_FORMAT = "pdf"
DPI = 300

CONDITIONS = ["intact", "word", "rest"]

# Match your existing colors
COND_COLORS = {
    "intact": "#7b1f83",   # purple
    "word": "#0b8f52",     # green
    "rest": "#222222",     # black/dark gray
}

# ------------------------------------------------------------
# Load full reconstruction summaries
# ------------------------------------------------------------

def expected_full_csv(base_dir, analysis, fit_scope):
    return (
        Path(base_dir)
        / f"msaa_condrank_decoding_outputs_{analysis}_{fit_scope}"
        / "full_summary.csv"
    )


def standardize_full_df(df, analysis, fit_scope):
    df = df.copy()

    if "analysis" not in df.columns:
        if "analysis_type" in df.columns:
            df["analysis"] = df["analysis_type"]
        else:
            df["analysis"] = analysis

    if "fit" not in df.columns:
        if "fit_scope" in df.columns:
            df["fit"] = df["fit_scope"]
        else:
            df["fit"] = fit_scope

    if "accuracy" not in df.columns:
        if "mean" in df.columns:
            df["accuracy"] = df["mean"]
        elif "mean_accuracy" in df.columns:
            df["accuracy"] = df["mean_accuracy"]
        else:
            raise KeyError(
                "Could not find accuracy column. Expected one of: "
                "accuracy, mean, mean_accuracy."
            )

    if "sem" not in df.columns:
        if "err" in df.columns:
            df["sem"] = df["err"]
        elif "sem_accuracy" in df.columns:
            df["sem"] = df["sem_accuracy"]
        else:
            df["sem"] = 0.0

    for c in ["analysis", "fit", "condition"]:
        df[c] = df[c].astype(str)

    df["K"] = df["K"].astype(int)

    return df


def load_full_outputs(base_dir=BASE_DIR):
    rows = []

    for analysis in ["spatial", "temporal"]:
        for fit_scope in ["across", "within"]:

            path = expected_full_csv(base_dir, analysis, fit_scope)

            if not path.exists():
                print("Missing:", path)
                continue

            df = pd.read_csv(path)
            df["source_csv"] = str(path)

            rows.append(
                standardize_full_df(
                    df,
                    analysis=analysis,
                    fit_scope=fit_scope,
                )
            )

    if not rows:
        raise FileNotFoundError("No full_summary.csv files found.")

    return pd.concat(rows, ignore_index=True)


full_df = load_full_outputs(BASE_DIR)

display(full_df.head())
print(full_df.groupby(["analysis", "fit", "condition"])["K"].nunique())


# ------------------------------------------------------------
# Plotting
# ------------------------------------------------------------

def plot_full_recon_panel(
    df,
    analysis="spatial",
    fit="across",
    ax=None,
):

    sub = df[
        (df["analysis"] == analysis)
        & (df["fit"] == fit)
    ].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}, fit={fit}")

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4.5))
    else:
        fig = ax.figure

    for condition in CONDITIONS:

        s = sub[sub["condition"] == condition].copy()

        if s.empty:
            continue

        s = s.sort_values("K")

        ax.errorbar(
            s["K"],
            s["accuracy"],
            yerr=s["sem"],
            color=COND_COLORS[condition],
            marker="o",
            markersize=5,
            linewidth=2.5,
            elinewidth=1.5,
            capsize=3,
            label=condition,
        )

    ax.set_title(
        f"{analysis} AA | {fit} | full reconstruction",
        fontsize=12
    )

    ax.set_xlabel("K", fontsize=11)
    ax.set_ylabel("Decoding accuracy", fontsize=11)

    ax.legend(frameon=False, fontsize=9, loc="upper left")

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    ax.tick_params(axis="both", labelsize=10)

    return fig, ax


def make_full_recon_figure(df, fit="across", save=True):

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5),
        sharey=True
    )

    plot_full_recon_panel(
        df,
        analysis="spatial",
        fit=fit,
        ax=axes[0],
    )

    plot_full_recon_panel(
        df,
        analysis="temporal",
        fit=fit,
        ax=axes[1],
    )

    # Match the uploaded figure style: no big suptitle
    fig.tight_layout()

    if save:
        out = FIG_DIR / f"full_recon_K_{fit}.{FIG_FORMAT}"
        fig.savefig(out, dpi=DPI, bbox_inches="tight")
        print("Saved:", out)

    return fig


# ------------------------------------------------------------
# Make across and within figures
# ------------------------------------------------------------

fig_across = make_full_recon_figure(full_df, fit="across", save=True)
plt.show()

fig_within = make_full_recon_figure(full_df, fit="within", save=True)
plt.show()

In [ ]:
# ============================================================
# Full reconstruction decoding figure
# WITHIN + ACROSS on same axes
# using 95% confidence intervals
#
# Style matched to original figure
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

BASE_DIR = Path(".")

FIG_DIR = Path("figures") / "full_reconstruction_decoding"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_FORMAT = "pdf"
DPI = 300

CONDITIONS = ["intact", "word", "rest"]

COND_COLORS = {
    "intact": "#7b1f83",   # purple
    "word": "#0b8f52",     # green
    "rest": "#222222",     # dark gray / black
}

FIT_STYLES = {
    "across": "-",
    "within": "--",
}

FIT_LABELS = {
    "across": "across",
    "within": "within",
}

# ------------------------------------------------------------
# Load full reconstruction summaries
# ------------------------------------------------------------

def expected_full_csv(base_dir, analysis, fit_scope):
    return (
        Path(base_dir)
        / f"msaa_condrank_decoding_outputs_{analysis}_{fit_scope}"
        / "full_summary.csv"
    )


def standardize_full_df(df, analysis, fit_scope):

    df = df.copy()

    if "accuracy" not in df.columns:

        if "mean" in df.columns:
            df["accuracy"] = df["mean"]

        elif "mean_accuracy" in df.columns:
            df["accuracy"] = df["mean_accuracy"]

        else:
            raise KeyError("Could not find accuracy column.")

    if "sem" not in df.columns:

        if "err" in df.columns:
            df["sem"] = df["err"]

        elif "sem_accuracy" in df.columns:
            df["sem"] = df["sem_accuracy"]

        else:
            df["sem"] = 0.0

    df["analysis"] = analysis
    df["fit"] = fit_scope

    df["K"] = df["K"].astype(int)

    return df


def load_full_outputs(base_dir=BASE_DIR):

    rows = []

    for analysis in ["spatial", "temporal"]:
        for fit_scope in ["across", "within"]:

            path = expected_full_csv(base_dir, analysis, fit_scope)

            if not path.exists():
                print("Missing:", path)
                continue

            df = pd.read_csv(path)

            rows.append(
                standardize_full_df(
                    df,
                    analysis=analysis,
                    fit_scope=fit_scope,
                )
            )

    if not rows:
        raise FileNotFoundError("No full_summary.csv files found.")

    return pd.concat(rows, ignore_index=True)


full_df = load_full_outputs(BASE_DIR)

display(full_df.head())


# ------------------------------------------------------------
# Plotting helper
# ------------------------------------------------------------

def plot_combined_panel(
    df,
    analysis="spatial",
    ax=None,
):

    sub = df[df["analysis"] == analysis].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}")

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        fig = ax.figure

    for condition in CONDITIONS:

        for fit in ["across", "within"]:

            s = sub[
                (sub["condition"] == condition)
                & (sub["fit"] == fit)
            ].copy()

            if s.empty:
                continue

            s = s.sort_values("K")

            # 95% confidence interval
            ci95 = 1.96 * s["sem"]

            ax.plot(
                s["K"],
                s["accuracy"],
                color=COND_COLORS[condition],
                linestyle=FIT_STYLES[fit],
                linewidth=2.5,
                marker="o",
                markersize=4,
                label=f"{condition} ({fit})",
            )

            ax.fill_between(
                s["K"],
                s["accuracy"] - ci95,
                s["accuracy"] + ci95,
                color=COND_COLORS[condition],
                alpha=0.15,
            )

    ax.set_title(
        f"{analysis} AA | full reconstruction",
        fontsize=13
    )

    ax.set_xlabel("K", fontsize=11)
    ax.set_ylabel("Decoding accuracy", fontsize=11)

    ax.tick_params(axis="both", labelsize=10)

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    return fig, ax


# ------------------------------------------------------------
# Generate figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    sharey=True
)

plot_combined_panel(
    full_df,
    analysis="spatial",
    ax=axes[0],
)

plot_combined_panel(
    full_df,
    analysis="temporal",
    ax=axes[1],
)

# ------------------------------------------------------------
# Build clean combined legend
# ------------------------------------------------------------

from matplotlib.lines import Line2D

legend_handles = []

# condition colors
for cond in CONDITIONS:

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color=COND_COLORS[cond],
            lw=3,
            label=cond,
        )
    )

# fit styles
for fit in ["across", "within"]:

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color="black",
            linestyle=FIT_STYLES[fit],
            lw=2.5,
            label=fit,
        )
    )

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=5,
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, 1.02),
)

fig.tight_layout(rect=[0, 0, 1, 0.95])

out = FIG_DIR / f"full_reconstruction_combined_within_across_ci95.{FIG_FORMAT}"

fig.savefig(
    out,
    dpi=DPI,
    bbox_inches="tight",
)

print("Saved:", out)

plt.show()

In [ ]:
plt.clf()

In [ ]:
# ============================================================
# Full reconstruction decoding figure
#
# ACROSS-CONDITION ONLY
# Spatial vs Temporal AA
# 95% confidence intervals
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

BASE_DIR = Path(".")

FIG_DIR = Path("figures") / "full_reconstruction_decoding"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_FORMAT = "pdf"
DPI = 300

CONDITIONS = ["intact", "word", "rest"]

COND_COLORS = {
    "intact": "#7b1f83",   # purple
    "word": "#0b8f52",     # green
    "rest": "#222222",     # black
}


# ------------------------------------------------------------
# Load full reconstruction summaries
# ------------------------------------------------------------

def expected_full_csv(base_dir, analysis):

    return (
        Path(base_dir)
        / f"msaa_condrank_decoding_outputs_{analysis}_across"
        / "full_summary.csv"
    )


def standardize_full_df(df, analysis):

    df = df.copy()

    if "accuracy" not in df.columns:

        if "mean" in df.columns:
            df["accuracy"] = df["mean"]

        elif "mean_accuracy" in df.columns:
            df["accuracy"] = df["mean_accuracy"]

        else:
            raise KeyError("Could not find accuracy column.")

    if "sem" not in df.columns:

        if "err" in df.columns:
            df["sem"] = df["err"]

        elif "sem_accuracy" in df.columns:
            df["sem"] = df["sem_accuracy"]

        else:
            df["sem"] = 0.0

    df["analysis"] = analysis

    df["K"] = df["K"].astype(int)

    return df


def load_full_outputs(base_dir=BASE_DIR):

    rows = []

    for analysis in ["spatial", "temporal"]:

        path = expected_full_csv(base_dir, analysis)

        if not path.exists():

            print("Missing:", path)

            continue

        df = pd.read_csv(path)

        rows.append(
            standardize_full_df(
                df,
                analysis=analysis,
            )
        )

    if not rows:

        raise FileNotFoundError(
            "No across-condition full_summary.csv files found."
        )

    return pd.concat(rows, ignore_index=True)


full_df = load_full_outputs(BASE_DIR)

display(full_df.head())


# ------------------------------------------------------------
# Plot helper
# ------------------------------------------------------------

def plot_panel(df, analysis="spatial", ax=None):

    sub = df[df["analysis"] == analysis].copy()

    if ax is None:

        fig, ax = plt.subplots(figsize=(6,5))

    else:

        fig = ax.figure

    for condition in CONDITIONS:

        s = sub[sub["condition"] == condition].copy()

        if s.empty:

            continue

        s = s.sort_values("K")

        ci95 = 1.96 * s["sem"]

        ax.plot(
            s["K"],
            s["accuracy"],
            color=COND_COLORS[condition],
            linewidth=2.5,
            marker="o",
            markersize=4,
            label=condition,
        )

        ax.fill_between(
            s["K"],
            s["accuracy"] - ci95,
            s["accuracy"] + ci95,
            color=COND_COLORS[condition],
            alpha=0.15,
        )

    ax.set_title(
        f"{analysis.capitalize()} AA",
        fontsize=18
    )

    ax.set_xlabel("Number of archetypes (K)", fontsize=14)

    ax.set_ylabel(
        "Across-condition decoding accuracy",
        fontsize=12
    )

    ax.tick_params(axis="both", labelsize=12)

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    return fig, ax


# ------------------------------------------------------------
# Generate figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14,5),
    sharey=True
)

plot_panel(
    full_df,
    analysis="spatial",
    ax=axes[0]
)

plot_panel(
    full_df,
    analysis="temporal",
    ax=axes[1]
)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

legend_handles = []

for cond in CONDITIONS:

    legend_handles.append(

        Line2D(
            [0],
            [0],
            color=COND_COLORS[cond],
            lw=3,
            label=cond,
        )
    )

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=3,
    frameon=False,
    fontsize=12,
    bbox_to_anchor=(0.5,1.02)
)

fig.tight_layout(rect=[0,0,1,0.95])


out = FIG_DIR / f"full_reconstruction_across_only_ci95.{FIG_FORMAT}"

fig.savefig(
    out,
    dpi=DPI,
    bbox_inches="tight",
)

print("Saved:", out)

plt.show()

In [ ]:
# ============================================================
# Zoomed full reconstruction figure
# x-axis zoom: normalized component ratio 0 -> 0.2
#
# SAME STYLE as previous figure
# ============================================================

# ------------------------------------------------------------
# Add normalized ratio
# ------------------------------------------------------------

ratio_df = full_df.copy()

ratio_df["ratio"] = ratio_df.apply(
    lambda r: r["K"] / DIMENSIONS[r["analysis"]],
    axis=1
)

# ------------------------------------------------------------
# Plotting helper
# ------------------------------------------------------------

def plot_zoomed_ratio_panel(
    df,
    analysis="spatial",
    ax=None,
    xlim=(0, 0.2),
):

    sub = df[df["analysis"] == analysis].copy()

    if sub.empty:
        raise ValueError(f"No rows for analysis={analysis}")

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        fig = ax.figure

    for condition in CONDITIONS:

        for fit in ["across", "within"]:

            s = sub[
                (sub["condition"] == condition)
                & (sub["fit"] == fit)
            ].copy()

            if s.empty:
                continue

            s = s.sort_values("ratio")

            # zoom range
            s = s[
                (s["ratio"] >= xlim[0])
                & (s["ratio"] <= xlim[1])
            ]

            if len(s) == 0:
                continue

            ci95 = 1.96 * s["sem"]

            ax.plot(
                s["ratio"],
                s["accuracy"],
                color=COND_COLORS[condition],
                linestyle=FIT_STYLES[fit],
                linewidth=2.5,
                marker="o",
                markersize=4,
            )

            ax.fill_between(
                s["ratio"],
                s["accuracy"] - ci95,
                s["accuracy"] + ci95,
                color=COND_COLORS[condition],
                alpha=0.15,
            )

            # annotate strongest decoding points
            top_idx = np.argsort(s["accuracy"].values)[-2:]

            for idx in top_idx:

                row = s.iloc[idx]

                ax.text(
                    row["ratio"],
                    row["accuracy"],
                    f"K={int(row['K'])}",
                    fontsize=7,
                    ha="left",
                    va="bottom",
                )

    ax.set_xlim(xlim)

    ax.set_title(
        f"{analysis} AA | full reconstruction",
        fontsize=13
    )

    ax.set_xlabel(
        "Normalized component ratio (K / D)",
        fontsize=11
    )

    ax.set_ylabel(
        "Decoding accuracy",
        fontsize=11
    )

    ax.tick_params(axis="both", labelsize=10)

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    return fig, ax


# ------------------------------------------------------------
# Generate zoomed figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    sharey=True
)

plot_zoomed_ratio_panel(
    ratio_df,
    analysis="spatial",
    ax=axes[0],
    xlim=(0, 0.2),
)

plot_zoomed_ratio_panel(
    ratio_df,
    analysis="temporal",
    ax=axes[1],
    xlim=(0, 0.2),
)

# ------------------------------------------------------------
# Shared legend
# ------------------------------------------------------------

from matplotlib.lines import Line2D

legend_handles = []

for cond in CONDITIONS:

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color=COND_COLORS[cond],
            lw=3,
            label=cond,
        )
    )

for fit in ["across", "within"]:

    legend_handles.append(
        Line2D(
            [0],
            [0],
            color="black",
            linestyle=FIT_STYLES[fit],
            lw=2.5,
            label=fit,
        )
    )

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=5,
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, 1.02),
)

fig.tight_layout(rect=[0, 0, 1, 0.95])

out = FIG_DIR / "full_reconstruction_combined_within_across_ci95_zoom_ratio_0_02.pdf"

fig.savefig(
    out,
    dpi=DPI,
    bbox_inches="tight",
)

print("Saved:", out)

plt.show()

In [ ]:
# ============================================================
# Export top raw-K annotations for exact matched-ratio panels
# ============================================================

def top_k_annotations_exact(df, analysis, fit, condition, top_n=3):
    heat, ratio_values, raw_k_values = build_exact_matched_heatmap(df, analysis, fit, condition)
    rows = []

    for i, m in enumerate(TOP_M_VALUES):
        row = heat[i, :]
        valid = np.where(np.isfinite(row))[0]
        if len(valid) == 0:
            continue

        top_idx = valid[np.argsort(row[valid])[-top_n:]][::-1]
        for rank, j in enumerate(top_idx, start=1):
            rows.append({
                "analysis": analysis,
                "fit": fit,
                "condition": condition,
                "top_m": int(m),
                "rank_within_top_m": rank,
                "K": int(raw_k_values[j]),
                "component_ratio": float(ratio_values[j]),
                "accuracy": float(row[j]),
            })

    return pd.DataFrame(rows)

all_ann = []
for condition in CONDITIONS:
    for analysis in ["spatial", "temporal"]:
        for fit in ["across", "within"]:
            all_ann.append(top_k_annotations_exact(matched_topm_df, analysis, fit, condition))

all_annotations_df = pd.concat(all_ann, ignore_index=True)
out_csv = TABLE_DIR / "exact_matched_ratio_top3_raw_K_annotations.csv"
all_annotations_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
display(all_annotations_df.head(30))

In [ ]:
# ============================================================
# Publication-style across-condition difference heatmaps
# Shared colorbar across all conditions
# Sparse axis labels + only selected K annotations
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

CONDITIONS_TO_PLOT = ["intact", "word", "rest"]

# annotate only these spatial K columns when available
ANNOTATE_SPATIAL_K = [14, 50, 88, 300, 700]

# annotate these top-m rows only
ANNOTATE_TOP_M = [5, 10, 15]

FIG_OUT = FIG_DIR / "spatial_minus_temporal_shared_colorbar_sparse_labels.pdf"

TITLE_MAP = {
    "intact": "Intact",
    "word": "Word-scrambled",
    "rest": "Rest",
}

# ------------------------------------------------------------
# Build heatmaps first so shared colorbar can be computed
# ------------------------------------------------------------

def make_difference_heat(condition):
    sub = custom_across_diff_df[
        custom_across_diff_df["condition"] == condition
    ].copy()

    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(x_table)), np.nan)

    for i, m in enumerate(top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            # mask saturated top-m values
            if m > spatial_K or m > temporal_K:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["spatial_K"] == spatial_K)
                & (sub["temporal_K"] == temporal_K),
                "difference_spatial_minus_temporal"
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, top_ms, x_table


heat_data = {
    condition: make_difference_heat(condition)
    for condition in CONDITIONS_TO_PLOT
}

global_max_abs = max(
    np.nanmax(np.abs(heat))
    for heat, _, _ in heat_data.values()
)

norm = TwoSlopeNorm(
    vmin=-global_max_abs,
    vcenter=0,
    vmax=global_max_abs,
)

print("Shared color scale:", -global_max_abs, global_max_abs)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=len(CONDITIONS_TO_PLOT),
    ncols=1,
    figsize=(15, 10),
    sharex=True,
    constrained_layout=True,
)

if len(CONDITIONS_TO_PLOT) == 1:
    axes = [axes]

for ax, condition in zip(axes, CONDITIONS_TO_PLOT):

    heat, top_ms, x_table = heat_data[condition]

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        norm=norm,
    )

    ax.set_title(
        TITLE_MAP.get(condition, condition),
        fontsize=17,
        fontweight="bold",
        pad=8,
    )

    ax.set_ylabel("top-$m$", fontsize=14)

    # y-axis labels
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms, fontsize=11)

    # sparse x-axis labels
    n_x = len(x_table)
    xtick_idx = np.unique(
        np.round(np.linspace(0, n_x - 1, 12)).astype(int)
    )
    ax.set_xlim(0, .95)
    ax.set_xticks(xtick_idx)
    ax.set_xticklabels(
        [f"{x_table.iloc[j]['matched_ratio']:.2f}" for j in xtick_idx],
        rotation=45,
        ha="right",
        fontsize=11,
    )

    ax.tick_params(axis="both", length=0)

    # --------------------------------------------------------
    # Annotate selected matched K values only
    # --------------------------------------------------------

    for spatial_K in ANNOTATE_SPATIAL_K:

        matches = np.where(
            x_table["spatial_K"].astype(int).values == spatial_K
        )[0]

        if len(matches) == 0:
            continue

        j = int(matches[0])
        temporal_K = int(x_table.iloc[j]["temporal_K"])

        for m in ANNOTATE_TOP_M:

            if m not in top_ms:
                continue

            i = top_ms.index(m)

            if not np.isfinite(heat[i, j]):
                continue

            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{spatial_K}\nT{temporal_K}",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold",
                color="white" if abs(val) > global_max_abs * 0.35 else "black",
            )

axes[-1].set_xlabel("Matched component ratio", fontsize=14)

# ------------------------------------------------------------
# Shared colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=axes,
    orientation="vertical",
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(
    "Spatial − temporal decoding",
    fontsize=14,
)

cbar.ax.tick_params(labelsize=12)

fig.suptitle(
    "Spatial minus temporal top-$m$ decoding across matched component ratios",
    fontsize=18,
    fontweight="bold",
)

fig.savefig(
    FIG_OUT,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", FIG_OUT)

plt.show()

In [ ]:
# ============================================================
# Publication-style across-condition difference heatmaps
# Shared colorbar across all conditions
# Sparse axis labels
# Annotates best decoded cells within selected top-m rows
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

CONDITIONS_TO_PLOT = ["intact", "word", "rest"]

FIG_OUT = FIG_DIR / "spatial_minus_temporal_shared_colorbar_best_cells.pdf"

TITLE_MAP = {
    "intact": "Intact",
    "word": "Word-scrambled",
    "rest": "Rest",
}

# Row-wise annotation settings
N_LABELS_PER_ROW = 5         # number of cells to annotate per top-m row
ANNOTATE_TOP_M = [5, 10, 15]  # sparse rows to annotate; use None for all rows
ANNOTATE_BY = "positive"      # "positive", "absolute", or "both"

# ------------------------------------------------------------
# Build heatmaps first so shared colorbar can be computed
# ------------------------------------------------------------

def make_difference_heat(condition):
    sub = custom_across_diff_df[
        custom_across_diff_df["condition"] == condition
    ].copy()

    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    top_ms = [m for m in TOP_M_VALUES if m in set(sub["top_m"])]

    heat = np.full((len(top_ms), len(x_table)), np.nan)

    for i, m in enumerate(top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            # mask invalid/saturated cases
            if m > spatial_K or m > temporal_K:
                continue

            vals = sub.loc[
                (sub["top_m"] == m)
                & (sub["spatial_K"] == spatial_K)
                & (sub["temporal_K"] == temporal_K),
                "difference_spatial_minus_temporal",
            ]

            if len(vals):
                heat[i, j] = vals.mean()

    return heat, top_ms, x_table


heat_data = {
    condition: make_difference_heat(condition)
    for condition in CONDITIONS_TO_PLOT
}

global_max_abs = max(
    np.nanmax(np.abs(heat))
    for heat, _, _ in heat_data.values()
)

norm = TwoSlopeNorm(
    vmin=-global_max_abs,
    vcenter=0,
    vmax=global_max_abs,
)

print("Shared color scale:", -global_max_abs, global_max_abs)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=len(CONDITIONS_TO_PLOT),
    ncols=1,
    figsize=(15, 10),
    sharex=True,
    constrained_layout=True,
)

if len(CONDITIONS_TO_PLOT) == 1:
    axes = [axes]

for ax, condition in zip(axes, CONDITIONS_TO_PLOT):

    heat, top_ms, x_table = heat_data[condition]

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        norm=norm,
    )

    ax.set_title(
        TITLE_MAP.get(condition, condition),
        fontsize=24,
        fontweight="bold",
        pad=8,
    )

    ax.set_ylabel("top-$m$", fontsize=18)

    # y-axis labels
    ax.set_yticks(np.arange(len(top_ms)))
    ax.set_yticklabels(top_ms, fontsize=14)

    # sparse x-axis labels
    n_x = len(x_table)
    xtick_idx = np.unique(
        np.round(np.linspace(0, n_x - 1, 12)).astype(int)
    )

    ax.set_xticks(xtick_idx)
    ax.set_xticklabels(
        [f"{x_table.iloc[j]['matched_ratio']:.2f}" for j in xtick_idx],
        rotation=45,
        ha="right",
        fontsize=14,
    )

    ax.tick_params(axis="both", length=0)

    # --------------------------------------------------------
    # Annotate best decoded cells within selected top-m rows
    # --------------------------------------------------------

    for i, m in enumerate(top_ms):

        if ANNOTATE_TOP_M is not None and m not in ANNOTATE_TOP_M:
            continue

        row_vals = heat[i, :]
        valid = np.where(np.isfinite(row_vals))[0]

        if len(valid) == 0:
            continue

        if ANNOTATE_BY == "positive":
            annotate_idx = valid[
                np.argsort(row_vals[valid])[-N_LABELS_PER_ROW:]
            ]

        elif ANNOTATE_BY == "absolute":
            annotate_idx = valid[
                np.argsort(np.abs(row_vals[valid]))[-N_LABELS_PER_ROW:]
            ]

        elif ANNOTATE_BY == "both":
            pos_idx = valid[
                np.argsort(row_vals[valid])[-N_LABELS_PER_ROW:]
            ]
            neg_idx = valid[
                np.argsort(row_vals[valid])[:N_LABELS_PER_ROW]
            ]
            annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

        else:
            raise ValueError(
                "ANNOTATE_BY must be 'positive', 'absolute', or 'both'."
            )

        for j in annotate_idx:

            spatial_K = int(x_table.iloc[j]["spatial_K"])
            temporal_K = int(x_table.iloc[j]["temporal_K"])
            val = heat[i, j]

            ax.text(
                j,
                i,
                f"S{spatial_K}\nT{temporal_K}",
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold",
                color="white" if abs(val) > global_max_abs * 0.35 else "black",
            )

axes[-1].set_xlabel("Matched component ratio", fontsize=14)

# ------------------------------------------------------------
# Shared colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=axes,
    orientation="vertical",
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(
    "Spatial − temporal decoding",
    fontsize=18,
)

cbar.ax.tick_params(labelsize=14)

fig.suptitle(
    "Spatial minus temporal top-$m$ decoding across matched component ratios",
    fontsize=24,
    fontweight="bold",
)

fig.savefig(
    FIG_OUT,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", FIG_OUT)

plt.show()

In [ ]:
# ============================================================
# Difference heatmaps using spatial top-m as the reference
#
# y-axis = spatial top-m
# temporal top-m is matched by relative component fraction:
#     temporal_m ~= spatial_m * (temporal_K / spatial_K)
#
# Shared colorbar across conditions
# Labels only on intact panel
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

CONDITIONS_TO_PLOT = ["intact", "word", "rest"]

FIG_OUT = FIG_DIR / "spatial_minus_temporal_relative_spatial_topm_shared_colorbar.pdf"

TITLE_MAP = {
    "intact": "Intact",
    "word": "Word-scrambled",
    "rest": "Rest",
}

LABEL_CONDITION = "intact"
N_LABELS_PER_ROW = 5
ANNOTATE_TOP_M = [3, 5, 10, 15]
ANNOTATE_BY = "positive"   # "positive", "absolute", or "both"

# ------------------------------------------------------------
# Robust column lookup
# ------------------------------------------------------------

def _get_col(df, options):
    for c in options:
        if c in df.columns:
            return c
    raise ValueError(f"None of these columns were found: {options}")

ANALYSIS_COL = _get_col(topm_df, ["analysis", "analysis_type"])
FIT_COL = _get_col(topm_df, ["fit", "fit_scope"])
ACC_COL = _get_col(topm_df, ["accuracy", "mean", "mean_accuracy"])

# ------------------------------------------------------------
# Available top-m lookup
# ------------------------------------------------------------

def nearest_available_topm(df, analysis, fit, condition, K, target_m):
    """
    Choose nearest available top_m for a given analysis / K / condition.
    This prevents asking for a top_m value that was not actually computed.
    """

    sub = df[
        (df[ANALYSIS_COL].astype(str) == analysis)
        & (df[FIT_COL].astype(str) == fit)
        & (df["condition"].astype(str) == condition)
        & (df["K"].astype(int) == int(K))
    ]

    if sub.empty:
        return None

    available = np.sort(sub["top_m"].astype(int).unique())

    # keep within legal range
    target_m = max(1, min(int(round(target_m)), int(K)))

    # nearest available computed top_m
    nearest = available[np.argmin(np.abs(available - target_m))]
    return int(nearest)

# ------------------------------------------------------------
# Build heatmap
# ------------------------------------------------------------

def make_difference_heat_spatial_relative(condition):
    """
    Build spatial minus temporal heatmap.

    The plotted row is spatial top-m.

    Temporal top-m is chosen by matching the fraction of components:
        temporal_m ~= spatial_m / spatial_K * temporal_K
    """

    x_table = CUSTOM_MATCHED_RATIO_TABLE.copy()
    spatial_top_ms = list(TOP_M_VALUES)

    heat = np.full((len(spatial_top_ms), len(x_table)), np.nan)

    temporal_m_used = np.full((len(spatial_top_ms), len(x_table)), np.nan)
    spatial_m_used = np.full((len(spatial_top_ms), len(x_table)), np.nan)

    for i, spatial_m in enumerate(spatial_top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            # y-axis is spatial top-m, so only mask if spatial_m > spatial_K
            if spatial_m > spatial_K:
                continue

            # match temporal top-m by relative fraction of components
            temporal_target_m = spatial_m * (temporal_K / spatial_K)

            temporal_m = nearest_available_topm(
                topm_df,
                analysis="temporal",
                fit="across",
                condition=condition,
                K=temporal_K,
                target_m=temporal_target_m,
            )

            if temporal_m is None:
                continue

            s = topm_df[
                (topm_df[ANALYSIS_COL].astype(str) == "spatial")
                & (topm_df[FIT_COL].astype(str) == "across")
                & (topm_df["condition"].astype(str) == condition)
                & (topm_df["K"].astype(int) == spatial_K)
                & (topm_df["top_m"].astype(int) == spatial_m)
            ]

            t = topm_df[
                (topm_df[ANALYSIS_COL].astype(str) == "temporal")
                & (topm_df[FIT_COL].astype(str) == "across")
                & (topm_df["condition"].astype(str) == condition)
                & (topm_df["K"].astype(int) == temporal_K)
                & (topm_df["top_m"].astype(int) == temporal_m)
            ]

            if len(s) == 0 or len(t) == 0:
                continue

            spatial_acc = float(s[ACC_COL].mean())
            temporal_acc = float(t[ACC_COL].mean())

            heat[i, j] = spatial_acc - temporal_acc
            spatial_m_used[i, j] = spatial_m
            temporal_m_used[i, j] = temporal_m

    return heat, spatial_top_ms, x_table, spatial_m_used, temporal_m_used

# ------------------------------------------------------------
# Build all heatmaps first for shared colorbar
# ------------------------------------------------------------

heat_data = {
    condition: make_difference_heat_spatial_relative(condition)
    for condition in CONDITIONS_TO_PLOT
}

global_max_abs = max(
    np.nanmax(np.abs(heat))
    for heat, _, _, _, _ in heat_data.values()
)

norm = TwoSlopeNorm(
    vmin=-global_max_abs,
    vcenter=0,
    vmax=global_max_abs,
)

print("Shared color scale:", -global_max_abs, global_max_abs)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=len(CONDITIONS_TO_PLOT),
    ncols=1,
    figsize=(15, 10),
    sharex=True,
    constrained_layout=True,
)

if len(CONDITIONS_TO_PLOT) == 1:
    axes = [axes]

for ax, condition in zip(axes, CONDITIONS_TO_PLOT):

    heat, spatial_top_ms, x_table, spatial_m_used, temporal_m_used = heat_data[condition]

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        norm=norm,
    )

    ax.set_title(
        TITLE_MAP.get(condition, condition),
        fontsize=24,
        fontweight="bold",
        pad=8,
    )

    ax.set_ylabel("spatial top-$m$", fontsize=18)

    ax.set_yticks(np.arange(len(spatial_top_ms)))
    ax.set_yticklabels(spatial_top_ms, fontsize=14)

    # sparse x-axis labels
    n_x = len(x_table)
    xtick_idx = np.unique(
        np.round(np.linspace(0, n_x - 1, 12)).astype(int)
    )

    ax.set_xticks(xtick_idx)
    ax.set_xticklabels(
        [f"{x_table.iloc[j]['matched_ratio']:.2f}" for j in xtick_idx],
        rotation=45,
        ha="right",
        fontsize=14,
    )

    ax.tick_params(axis="both", length=0)

    # --------------------------------------------------------
    # Annotate best cells only for intact
    # --------------------------------------------------------

    if condition == LABEL_CONDITION:

        for i, spatial_m in enumerate(spatial_top_ms):

            if ANNOTATE_TOP_M is not None and spatial_m not in ANNOTATE_TOP_M:
                continue

            row_vals = heat[i, :]
            valid = np.where(np.isfinite(row_vals))[0]

            if len(valid) == 0:
                continue

            if ANNOTATE_BY == "positive":
                annotate_idx = valid[
                    np.argsort(row_vals[valid])[-N_LABELS_PER_ROW:]
                ]

            elif ANNOTATE_BY == "absolute":
                annotate_idx = valid[
                    np.argsort(np.abs(row_vals[valid]))[-N_LABELS_PER_ROW:]
                ]

            elif ANNOTATE_BY == "both":
                pos_idx = valid[
                    np.argsort(row_vals[valid])[-N_LABELS_PER_ROW:]
                ]
                neg_idx = valid[
                    np.argsort(row_vals[valid])[:N_LABELS_PER_ROW]
                ]
                annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

            else:
                raise ValueError(
                    "ANNOTATE_BY must be 'positive', 'absolute', or 'both'."
                )

            for j in annotate_idx:

                spatial_K = int(x_table.iloc[j]["spatial_K"])
                temporal_K = int(x_table.iloc[j]["temporal_K"])
                temporal_m = int(temporal_m_used[i, j])
                val = heat[i, j]

                label = f"S{spatial_K}\nT{temporal_K}"
                # optional more explicit label:
                # label = f"S{spatial_K}\nT{temporal_K}\n$m_T$={temporal_m}"

                ax.text(
                    j,
                    i,
                    label,
                    ha="center",
                    va="center",
                    fontsize=12,
                    fontweight="bold",
                    color="black" if abs(val) > global_max_abs * 0.35 else "black",
                )

axes[-1].set_xlabel("Matched component ratio", fontsize=18)

# ------------------------------------------------------------
# Shared colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=axes,
    orientation="vertical",
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(
    "Spatial − temporal decoding",
    fontsize=18,
)

cbar.ax.tick_params(labelsize=18)

fig.suptitle(
    "Spatial minus temporal decoding using spatial top-$m$ reference",
    fontsize=24,
    fontweight="bold",
)

fig.savefig(
    FIG_OUT,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", FIG_OUT)

plt.show()

In [ ]:
# ============================================================
# Publication-style difference heatmaps
# Spatial top-m is the y-axis reference
# Temporal top-m is matched by relative component fraction
#
# IMPORTANT:
# - No binning
# - No averaging
# - No interpolation
# - X-axis is thinned by selecting a subset of existing matched pairs
# - Forces spatial K=50 and K=88 to be retained
# - Shared colorbar across all conditions
# - Labels only on intact panel
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

CONDITIONS_TO_PLOT = ["intact", "word", "rest"]

FIG_OUT = FIG_DIR / "spatial_minus_temporal_relative_spatial_topm_thinned_x_shared_colorbar.pdf"

TITLE_MAP = {
    "intact": "Intact",
    "word": "Word-scrambled",
    "rest": "Rest",
}

# Number of x positions to keep, roughly evenly spaced
N_X_KEEP = 22

# Force these spatial K values to remain on x-axis
FORCE_SPATIAL_K = [10, 50, 88]

# Annotation settings
LABEL_CONDITION = "intact"
N_LABELS_PER_ROW = 5
ANNOTATE_TOP_M = [5, 10, 15]
ANNOTATE_BY = "positive"   # "positive", "absolute", or "both"

# ------------------------------------------------------------
# Robust column lookup
# ------------------------------------------------------------

def _get_col(df, options):
    for c in options:
        if c in df.columns:
            return c
    raise ValueError(f"None of these columns were found: {options}")

ANALYSIS_COL = _get_col(topm_df, ["analysis", "analysis_type"])
FIT_COL = _get_col(topm_df, ["fit", "fit_scope"])
ACC_COL = _get_col(topm_df, ["accuracy", "mean", "mean_accuracy"])

# ------------------------------------------------------------
# Available top-m lookup
# ------------------------------------------------------------

def nearest_available_topm(df, analysis, fit, condition, K, target_m):
    sub = df[
        (df[ANALYSIS_COL].astype(str) == analysis)
        & (df[FIT_COL].astype(str) == fit)
        & (df["condition"].astype(str) == condition)
        & (df["K"].astype(int) == int(K))
    ]

    if sub.empty:
        return None

    available = np.sort(sub["top_m"].astype(int).unique())
    target_m = max(1, min(int(round(target_m)), int(K)))

    nearest = available[np.argmin(np.abs(available - target_m))]
    return int(nearest)

# ------------------------------------------------------------
# Thin x-axis by selecting actual matched-pair columns
# ------------------------------------------------------------

def thin_x_table_existing_pairs(x_table, n_keep=22, force_spatial_K=(50, 88)):
    """
    Select a subset of existing matched-pair columns.
    No averaging, no interpolation.

    Keeps approximately evenly spaced columns plus forced spatial K values.
    """

    x_table = x_table.reset_index(drop=True).copy()
    n = len(x_table)

    # evenly spaced existing indices
    keep_idx = set(np.round(np.linspace(0, n - 1, n_keep)).astype(int).tolist())

    # force specific spatial K values
    for k in force_spatial_K:
        matches = np.where(x_table["spatial_K"].astype(int).values == int(k))[0]
        for idx in matches:
            keep_idx.add(int(idx))

    keep_idx = sorted(keep_idx)

    return x_table.iloc[keep_idx].reset_index(drop=True)

# ------------------------------------------------------------
# Build spatial-reference difference heatmap
# ------------------------------------------------------------

def make_difference_heat_spatial_relative(condition, x_table):
    """
    Build spatial minus temporal heatmap.

    y-axis is spatial top-m.
    Temporal top-m is matched by relative component fraction:
        temporal_m ~= spatial_m * temporal_K / spatial_K
    """

    spatial_top_ms = list(TOP_M_VALUES)

    heat = np.full((len(spatial_top_ms), len(x_table)), np.nan)
    spatial_m_used = np.full((len(spatial_top_ms), len(x_table)), np.nan)
    temporal_m_used = np.full((len(spatial_top_ms), len(x_table)), np.nan)

    for i, spatial_m in enumerate(spatial_top_ms):
        for j, r in x_table.iterrows():

            spatial_K = int(r["spatial_K"])
            temporal_K = int(r["temporal_K"])

            # y-axis is spatial top-m; only mask if spatial_m > spatial_K
            if spatial_m > spatial_K:
                continue

            temporal_target_m = spatial_m * (temporal_K / spatial_K)

            temporal_m = nearest_available_topm(
                topm_df,
                analysis="temporal",
                fit="across",
                condition=condition,
                K=temporal_K,
                target_m=temporal_target_m,
            )

            if temporal_m is None:
                continue

            s = topm_df[
                (topm_df[ANALYSIS_COL].astype(str) == "spatial")
                & (topm_df[FIT_COL].astype(str) == "across")
                & (topm_df["condition"].astype(str) == condition)
                & (topm_df["K"].astype(int) == spatial_K)
                & (topm_df["top_m"].astype(int) == spatial_m)
            ]

            t = topm_df[
                (topm_df[ANALYSIS_COL].astype(str) == "temporal")
                & (topm_df[FIT_COL].astype(str) == "across")
                & (topm_df["condition"].astype(str) == condition)
                & (topm_df["K"].astype(int) == temporal_K)
                & (topm_df["top_m"].astype(int) == temporal_m)
            ]

            if len(s) == 0 or len(t) == 0:
                continue

            spatial_acc = float(s[ACC_COL].mean())
            temporal_acc = float(t[ACC_COL].mean())

            heat[i, j] = spatial_acc - temporal_acc
            spatial_m_used[i, j] = spatial_m
            temporal_m_used[i, j] = temporal_m

    return heat, spatial_top_ms, x_table, spatial_m_used, temporal_m_used

# ------------------------------------------------------------
# Create thinned x-table
# ------------------------------------------------------------

X_TABLE_THIN = thin_x_table_existing_pairs(
    CUSTOM_MATCHED_RATIO_TABLE.copy(),
    n_keep=N_X_KEEP,
    force_spatial_K=FORCE_SPATIAL_K,
)

print("Keeping x-axis matched pairs:")
display(
    X_TABLE_THIN[
        ["spatial_K", "temporal_K", "spatial_ratio", "temporal_ratio", "matched_ratio"]
    ]
)

# ------------------------------------------------------------
# Build heatmaps
# ------------------------------------------------------------

heat_data = {
    condition: make_difference_heat_spatial_relative(condition, X_TABLE_THIN)
    for condition in CONDITIONS_TO_PLOT
}

global_max_abs = max(
    np.nanmax(np.abs(heat))
    for heat, _, _, _, _ in heat_data.values()
)

norm = TwoSlopeNorm(
    vmin=-global_max_abs,
    vcenter=0,
    vmax=global_max_abs,
)

print("Shared color scale:", -global_max_abs, global_max_abs)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=len(CONDITIONS_TO_PLOT),
    ncols=1,
    figsize=(14, 9.5),
    sharex=True,
    constrained_layout=True,
)

if len(CONDITIONS_TO_PLOT) == 1:
    axes = [axes]

for ax, condition in zip(axes, CONDITIONS_TO_PLOT):

    heat, spatial_top_ms, x_table, spatial_m_used, temporal_m_used = heat_data[condition]

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        norm=norm,
    )

    ax.set_title(
        TITLE_MAP.get(condition, condition),
        fontsize=17,
        fontweight="bold",
        pad=8,
    )

    ax.set_ylabel("spatial top-$m$", fontsize=14)

    ax.set_yticks(np.arange(len(spatial_top_ms)))
    ax.set_yticklabels(spatial_top_ms, fontsize=11)

    ax.set_xticks(np.arange(len(x_table)))
    ax.set_xticklabels(
        [f"{r:.2f}" for r in x_table["matched_ratio"]],
        rotation=45,
        ha="right",
        fontsize=10,
    )

    ax.tick_params(axis="both", length=0)

    # --------------------------------------------------------
    # Annotate best cells only for intact
    # --------------------------------------------------------

    if condition == LABEL_CONDITION:

        for i, spatial_m in enumerate(spatial_top_ms):

            if ANNOTATE_TOP_M is not None and spatial_m not in ANNOTATE_TOP_M:
                continue

            row_vals = heat[i, :]
            valid = np.where(np.isfinite(row_vals))[0]

            if len(valid) == 0:
                continue

            if ANNOTATE_BY == "positive":
                annotate_idx = valid[
                    np.argsort(row_vals[valid])[-N_LABELS_PER_ROW:]
                ]

            elif ANNOTATE_BY == "absolute":
                annotate_idx = valid[
                    np.argsort(np.abs(row_vals[valid]))[-N_LABELS_PER_ROW:]
                ]

            elif ANNOTATE_BY == "both":
                pos_idx = valid[
                    np.argsort(row_vals[valid])[-N_LABELS_PER_ROW:]
                ]
                neg_idx = valid[
                    np.argsort(row_vals[valid])[:N_LABELS_PER_ROW]
                ]
                annotate_idx = np.unique(np.concatenate([pos_idx, neg_idx]))

            else:
                raise ValueError(
                    "ANNOTATE_BY must be 'positive', 'absolute', or 'both'."
                )

            for j in annotate_idx:

                spatial_K = int(x_table.iloc[j]["spatial_K"])
                temporal_K = int(x_table.iloc[j]["temporal_K"])
                val = heat[i, j]

                label = f"S{spatial_K}\nT{temporal_K}"

                ax.text(
                    j,
                    i,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                    color="black" if abs(val) > global_max_abs * 0.35 else "black",
                )

axes[-1].set_xlabel("Matched component ratio", fontsize=14)

# ------------------------------------------------------------
# Shared colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=axes,
    orientation="vertical",
    fraction=0.025,
    pad=0.02,
)

cbar.set_label(
    "Spatial − temporal decoding",
    fontsize=14,
)

cbar.ax.tick_params(labelsize=11)

fig.suptitle(
    "Spatial minus temporal decoding using spatial top-$m$ reference",
    fontsize=18,
    fontweight="bold",
)

fig.savefig(
    FIG_OUT,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", FIG_OUT)

plt.show()

## Notes

- No interpolation is performed.
- Only exact matched component ratios are shown.
- Each ratio column has:
  - one spatial raw `K`
  - one temporal raw `K`
- The same matched ratio columns are used for spatial/temporal and within/across panels.
- The raw `K` annotations refer to the raw model order for that panel, so the same ratio has different raw `K` in spatial and temporal panels.